# PHSX 256 Topic 15: Numerical Integration of Ordinary Differential Equations

---

Physics is largely the science of *change*. Newton's second law, Maxwell's equations, Schrödinger's equation — all of these are **ordinary differential equations** (ODEs) describing how physical quantities evolve in time. Most of them have no closed-form analytic solution, so we must solve them numerically.

---

## Standard Imports and Live-Plot Helper

All examples use a `LivePlot` helper that works reliably in JupyterLite/Pyodide. Run this cell first.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

# Imports below only needed for animated plots
import piplite
await piplite.install(['ipympl', 'matplotlib'])

import asyncio
import matplotlib
matplotlib.use('agg')          # non-interactive backend — required for LivePlot
from IPython.display import display, clear_output
import ipywidgets as widgets

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12,
    'lines.linewidth': 2,
})

# ── Live-plot helper (works in JupyterLite / Pyodide) ────────────────────────

class LivePlot:
    """
    Context manager for live-updating matplotlib plots in JupyterLite.

    Usage:
        with LivePlot() as lp:
            lp.fig, lp.ax  # set up your plot here
            ...
            await lp.update()   # call inside an async loop to refresh the display
    """
    def __init__(self, figsize=(9, 4), min_interval=0.05):
        self.figsize = figsize
        self.min_interval = min_interval   # minimum seconds between renders
        self._last_draw = 0

    def __enter__(self):
        self._out = widgets.Output()
        display(self._out)
        self.fig, self.ax = plt.subplots(figsize=self.figsize)
        return self

    def __exit__(self, *args):
        self._render()   # always show the final frame
        plt.close(self.fig)

    def _render(self):
        with self._out:
            clear_output(wait=True)
            display(self.fig)

    async def update(self, force=False):
        """Re-render the figure. Skips frames faster than min_interval."""
        now = asyncio.get_event_loop().time()
        if force or (now - self._last_draw) >= self.min_interval:
            self._render()
            self._last_draw = now
            await asyncio.sleep(0)   # yield to the browser event loop

print("Imports and LivePlot ready.")

---
## Part 1 — First-Order ODEs and the Euler Method

### What is a first-order ODE?

The simplest ODE relates a function $y(t)$ to its first derivative:

$$\frac{dy}{dt} = f(t, y)$$

Given an **initial condition** $y(t_0) = y_0$, we want to find $y(t)$ for all $t > t_0$. The function $f$ is the **right-hand side** — it tells us the *rate of change* of $y$ at any point $(t, y)$.

**A few examples from physics:**

| Physical situation | ODE |
|---|---|
| Radioactive decay | $dN/dt = -\lambda N$ |
| Newton's cooling | $dT/dt = -k(T - T_{\rm env})$ |
| RC circuit charging | $dV/dt = (V_{\rm in} - V)/RC$ |
| Free fall (velocity only) | $dv/dt = -g$ |

All of these have the same structure: they tell us the slope $dy/dt$ as a function of the current state.

### The Euler method

The **Euler method** is the simplest numerical integrator. The key idea is:

> *If I know the slope $dy/dt$ at time $t_n$, I can estimate $y$ at time $t_{n+1} = t_n + h$ by following that slope for a small step $h$.*

$$\boxed{y_{n+1} = y_n + h\,f(t_n, y_n)}$$

This is exactly the definition of the derivative rearranged:
$$\frac{dy}{dt} \approx \frac{y(t+h) - y(t)}{h} \implies y(t+h) \approx y(t) + h \cdot y'(t)$$

**Truncation error:** the Taylor expansion of $y(t+h)$ is
$$y(t+h) = y(t) + h\,y'(t) + \frac{h^2}{2}y''(t) + \cdots$$
By using only the first two terms, we discard $O(h^2)$ per step. Accumulated over $T/h$ steps, the **global error** is $O(h)$ — halving $h$ halves the error. This is a **first-order** method.

In [ ]:
# ── Euler integrator for scalar (1-D) ODEs ───────────────────────────────────

def euler_step(f, t, y, h):
    # Single Euler step: y_{n+1} = y_n + h * f(t_n, y_n)
    return y + h * f(t, y)

def euler_scalar(f, t_span, y0, h):
    """
    Integrate dy/dt = f(t, y) from t_span[0] to t_span[1] using Euler's method.
    y0 is a scalar initial condition.
    Returns arrays t, y.
    """
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = y[i] + h * f(t[i], y[i])
    return t, y

print("Scalar Euler integrator defined.")

### Example 1: Radioactive Decay

The simplest first-order ODE in physics:
$$\frac{dN}{dt} = -\lambda N, \qquad N(0) = N_0$$

Exact solution: $N(t) = N_0 e^{-\lambda t}$.

This is a perfect first test — we know the answer and can see how Euler compares.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: watch Euler build the solution step by step ─────────────────
# h=0.5 so steps are visible; exact solution shown as reference

lam = 0.5    # decay constant (1/s)
N0  = 1000.0 # initial number of nuclei
T   = 10.0   # simulate 10 seconds (5 half-lives)

def decay_rhs(t, N):
    return -lam * N

t_exact = np.linspace(0, T, 500)
N_exact = N0 * np.exp(-lam * t_exact)
h_anim = 0.5
t_anim = np.arange(0, T + h_anim, h_anim)
N_anim = np.zeros(len(t_anim))
N_anim[0] = N0

with LivePlot(figsize=(9, 4)) as lp:
    lp.ax.plot(t_exact, N_exact, 'k--', lw=2, alpha=0.5, label='Exact')
    line_anim, = lp.ax.plot([], [], 'o-', color='tomato', lw=2,
                            ms=7, label=f'Euler h={h_anim}')
    dot, = lp.ax.plot([], [], 'o', color='tomato', ms=12, zorder=5)
    lp.ax.set_xlim(0, T); lp.ax.set_ylim(0, N0 * 1.05)
    lp.ax.set_xlabel('Time (s)'); lp.ax.set_ylabel('$N$')
    lp.ax.set_title('Radioactive Decay: Euler steps building up')
    lp.ax.legend()
    
    for i_step in range(1, len(t_anim)):
        N_anim[i_step] = euler_step(decay_rhs, t_anim[i_step-1], N_anim[i_step-1], h_anim)
        line_anim.set_data(t_anim[:i_step+1], N_anim[:i_step+1])
        dot.set_data([t_anim[i_step]], [N_anim[i_step]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Radioactive decay: dN/dt = -lambda * N
lam = 0.5    # decay constant (1/s)
N0  = 1000.0 # initial number of nuclei
T   = 10.0   # simulate 10 seconds (5 half-lives)

def decay_rhs(t, N):
    return -lam * N

t_exact = np.linspace(0, T, 500)
N_exact = N0 * np.exp(-lam * t_exact)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_exact, N_exact, 'k--', lw=2, label='Exact $N_0 e^{-\\lambda t}$')

colors = ['tomato', 'steelblue', 'seagreen']
for h, col in zip([2.0, 0.5, 0.1], colors):
    t_n, N_n = euler_scalar(decay_rhs, (0, T), N0, h)
    ax.plot(t_n, N_n, 'o-', color=col, ms=4, lw=1.5, label=f'Euler h={h}')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Number of nuclei $N$')
ax.set_title('Radioactive Decay: Euler Method vs. Exact Solution')
ax.legend()
plt.tight_layout()
plt.show()

**Observations:** With a large step $h=2$, the Euler method is visibly inaccurate. With $h=0.1$, it closely tracks the exact exponential.

Notice something important: the error gets smaller as $h$ decreases. Let's measure this quantitatively.

In [ ]:
plt.close('all')
%matplotlib inline

# Error scaling for radioactive decay
h_vals = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
errors = []
for h in h_vals:
    t_n, N_n = euler_scalar(decay_rhs, (0, T), N0, h)
    # Error at t = T
    N_true = N0 * np.exp(-lam * T)
    errors.append(abs(N_n[-1] - N_true))

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(h_vals, errors, 'o-', color='steelblue', ms=8, label='Euler error')
h_arr = np.array(h_vals)
ax.loglog(h_arr, 50 * h_arr, 'k--', label='$O(h)$ reference')
ax.set_xlabel('Step size $h$')
ax.set_ylabel('Absolute error at $t=10$ s')
ax.set_title('Euler Method: First-Order Error Scaling')
ax.legend()
plt.tight_layout()
plt.show()

print("When h is halved, error also roughly halves: that's first-order accuracy.")

### Example 2: Newton's Law of Cooling

A hot object at temperature $T$ cools toward the ambient temperature $T_{\rm env}$:
$$\frac{dT}{dt} = -k(T - T_{\rm env}), \qquad T(0) = T_0$$

Exact solution: $T(t) = T_{\rm env} + (T_0 - T_{\rm env})\,e^{-kt}$.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: Euler building Newton's cooling solution step by step ────────
T_env = 20.0   # ambient temperature (°C)
T0_cool = 90.0 # initial temperature
k_cool  = 0.3  # cooling constant

def cooling_rhs(t, T):
    return -k_cool * (T - T_env)

t_c = np.linspace(0, 15, 500)
T_exact_cool = T_env + (T0_cool - T_env) * np.exp(-k_cool * t_c)

h_anim_c = 1.0
t_anim_c = np.arange(0, 15 + h_anim_c, h_anim_c)
T_anim_c = np.zeros(len(t_anim_c))
T_anim_c[0] = T0_cool

with LivePlot(figsize=(9, 4)) as lp:
    lp.ax.plot(t_c, T_exact_cool, 'k--', lw=2, alpha=0.5, label='Exact')
    lp.ax.axhline(T_env, color='gray', lw=1, ls=':', label=f'$T_{{env}}$')
    line_c, = lp.ax.plot([], [], 'o-', color='steelblue', lw=2, ms=7, label=f'Euler h={h_anim_c}')
    dot_c,  = lp.ax.plot([], [], 'o', color='steelblue', ms=12, zorder=5)
    lp.ax.set_xlim(0, 15); lp.ax.set_ylim(T_env - 5, T0_cool + 5)
    lp.ax.set_xlabel('Time (s)'); lp.ax.set_ylabel('Temperature (°C)')
    lp.ax.set_title("Newton's Cooling: Euler steps building up")
    lp.ax.legend()
    
    for i_step in range(1, len(t_anim_c)):
        T_anim_c[i_step] = T_anim_c[i_step-1] + h_anim_c * cooling_rhs(t_anim_c[i_step-1], T_anim_c[i_step-1])
        line_c.set_data(t_anim_c[:i_step+1], T_anim_c[:i_step+1])
        dot_c.set_data([t_anim_c[i_step]], [T_anim_c[i_step]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Newton's cooling
T_env = 20.0   # ambient temperature (°C)
T0_cool = 90.0 # initial temperature
k_cool  = 0.3  # cooling constant

def cooling_rhs(t, T):
    return -k_cool * (T - T_env)

t_c = np.linspace(0, 15, 500)
T_exact_cool = T_env + (T0_cool - T_env) * np.exp(-k_cool * t_c)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_c, T_exact_cool, 'k--', lw=2, label='Exact')
ax.axhline(T_env, color='gray', lw=1, ls=':', label=f'$T_{{env}}={T_env}°C$')

for h, col in zip([1.0, 0.5, 0.1], ['tomato', 'steelblue', 'seagreen']):
    t_n, T_n = euler_scalar(cooling_rhs, (0, 15), T0_cool, h)
    ax.plot(t_n, T_n, 'o-', color=col, ms=4, lw=1.5, label=f'Euler h={h}')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature (°C)')
ax.set_title("Newton's Law of Cooling")
ax.legend()
plt.tight_layout()
plt.show()

### Example 3: RC Circuit Charging

A capacitor in series with a resistor charges toward the supply voltage $V_{\rm in}$:
$$\frac{dV}{dt} = \frac{V_{\rm in} - V}{RC}$$

Exact solution: $V(t) = V_{\rm in}(1 - e^{-t/RC})$.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: RC circuit charging — Euler step by step ─────────────────────
Vin = 5.0   # supply voltage (V)
RC  = 1.0   # time constant (s)

def rc_rhs(t, V):
    return (Vin - V) / RC

t_rc = np.linspace(0, 5*RC, 500)
V_exact = Vin * (1 - np.exp(-t_rc / RC))

h_anim_rc = 0.5
t_anim_rc = np.arange(0, 5*RC + h_anim_rc, h_anim_rc)
V_anim_rc = np.zeros(len(t_anim_rc))
V_anim_rc[0] = 0.0

with LivePlot(figsize=(9, 4)) as lp:
    lp.ax.plot(t_rc, V_exact, 'k--', lw=2, alpha=0.5, label='Exact')
    lp.ax.axhline(Vin, color='gray', lw=1, ls=':', label=f'$V_{{in}}={Vin}$ V')
    line_rc, = lp.ax.plot([], [], 'o-', color='seagreen', lw=2, ms=7, label=f'Euler h={h_anim_rc}')
    dot_rc,  = lp.ax.plot([], [], 'o', color='seagreen', ms=12, zorder=5)
    lp.ax.set_xlim(0, 5*RC); lp.ax.set_ylim(-0.1, Vin * 1.1)
    lp.ax.set_xlabel('Time (s)'); lp.ax.set_ylabel('Voltage (V)')
    lp.ax.set_title('RC Circuit Charging: Euler steps building up')
    lp.ax.legend()
    
    for i_step in range(1, len(t_anim_rc)):
        V_anim_rc[i_step] = V_anim_rc[i_step-1] + h_anim_rc * rc_rhs(t_anim_rc[i_step-1], V_anim_rc[i_step-1])
        line_rc.set_data(t_anim_rc[:i_step+1], V_anim_rc[:i_step+1])
        dot_rc.set_data([t_anim_rc[i_step]], [V_anim_rc[i_step]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# RC circuit charging
Vin = 5.0   # supply voltage (V)
RC  = 1.0   # time constant (s)

def rc_rhs(t, V):
    return (Vin - V) / RC

t_rc = np.linspace(0, 5*RC, 500)
V_exact = Vin * (1 - np.exp(-t_rc / RC))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_rc, V_exact, 'k--', lw=2, label='Exact')
ax.axhline(Vin, color='gray', lw=1, ls=':', label=f'$V_{{in}}={Vin}$ V')

for h, col in zip([1.0, 0.5, 0.1], ['tomato', 'steelblue', 'seagreen']):
    t_n, V_n = euler_scalar(rc_rhs, (0, 5*RC), 0.0, h)
    ax.plot(t_n, V_n, 'o-', color=col, ms=4, lw=1.5, label=f'Euler h={h}')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage $V$ (V)')
ax.set_title('RC Circuit Charging')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 2 — Second-Order ODEs: Systems of First-Order Equations

### Reducing order: the state vector

Newton's second law $F = ma$ is a *second-order* ODE:
$$m\ddot{x} = F(x, \dot{x}, t)$$

Our Euler integrator only handles first-order equations. The solution is to **introduce a new variable** for the velocity $v = \dot{x}$, turning one second-order ODE into two coupled first-order ODEs:

$$\frac{dx}{dt} = v, \qquad \frac{dv}{dt} = \frac{F(x, v, t)}{m}$$

We bundle the variables into a **state vector** $\mathbf{y} = (x, v)$. The right-hand side becomes a vector function:

$$\frac{d\mathbf{y}}{dt} = \begin{pmatrix} v \\ F/m \end{pmatrix} = f(t, \mathbf{y})$$

The Euler formula $\mathbf{y}_{n+1} = \mathbf{y}_n + h\,f(t_n, \mathbf{y}_n)$ works identically — we just replace scalars with vectors.

In [ ]:
# ── Vector Euler integrator (works for any system of first-order ODEs) ────────

def euler_step(f, t, y, h):
    """Single Euler step. y is a NumPy array."""
    return y + h * f(t, y)

def euler_integrate(f, t_span, y0, h):
    """
    Integrate dy/dt = f(t, y) from t_span[0] to t_span[1].
    y0 is a NumPy array (state vector).
    Returns arrays t (shape N) and y (shape N x len(y0)).
    """
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = euler_step(f, t[i], y[i], h)
    return t, y

print("Vector Euler integrator defined.")

### Example 4: Free Fall

The simplest second-order ODE: a ball dropped from rest under gravity (no air resistance).

$$\ddot{y} = -g \implies \frac{dy}{dt} = v, \quad \frac{dv}{dt} = -g$$

Exact solution: $y(t) = y_0 - \frac{1}{2}g t^2$.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: Free fall — watch the Euler steps accumulate ─────────────────
g_ff = 9.81   # m/s^2
y0_ff = 100.0  # initial height (m)
v0_ff = 0.0    # starts at rest

def freefall_rhs(t, state):
    y, v = state
    return np.array([v, -g_ff])

T_fall = np.sqrt(2 * y0_ff / g_ff)  # exact time to hit ground

t_ex_ff = np.linspace(0, T_fall, 300)
y_ex_ff = y0_ff - 0.5 * g_ff * t_ex_ff**2

h_anim_ff = 0.5
t_anim_ff = np.arange(0, T_fall * 1.05 + h_anim_ff, h_anim_ff)
y_anim_ff = np.zeros((len(t_anim_ff), 2))
y_anim_ff[0] = np.array([y0_ff, v0_ff])

with LivePlot(figsize=(9, 4)) as lp:
    lp.ax.plot(t_ex_ff, y_ex_ff, 'k--', lw=2, alpha=0.5, label='Exact')
    lp.ax.axhline(0, color='saddlebrown', lw=2, alpha=0.5, label='Ground')
    line_ff, = lp.ax.plot([], [], 'o-', color='steelblue', lw=2, ms=7, label=f'Euler h={h_anim_ff}')
    dot_ff,  = lp.ax.plot([], [], 'o', color='steelblue', ms=12, zorder=5)
    lp.ax.set_xlim(0, T_fall * 1.05); lp.ax.set_ylim(-10, y0_ff * 1.05)
    lp.ax.set_xlabel('Time (s)'); lp.ax.set_ylabel('Height (m)')
    lp.ax.set_title('Free Fall: Euler steps building up')
    lp.ax.legend()
    
    for i_step in range(1, len(t_anim_ff)):
        y_anim_ff[i_step] = euler_step(freefall_rhs, t_anim_ff[i_step-1], y_anim_ff[i_step-1], h_anim_ff)
        if y_anim_ff[i_step, 0] < 0:
            break
        line_ff.set_data(t_anim_ff[:i_step+1], y_anim_ff[:i_step+1, 0])
        dot_ff.set_data([t_anim_ff[i_step]], [y_anim_ff[i_step, 0]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Free fall: state = [y, v]
g_ff = 9.81   # m/s^2
y0_ff = 100.0  # initial height (m)
v0_ff = 0.0    # starts at rest

def freefall_rhs(t, state):
    y, v = state
    return np.array([v, -g_ff])

T_fall = np.sqrt(2 * y0_ff / g_ff)  # exact time to hit ground

t_ex_ff = np.linspace(0, T_fall, 300)
y_ex_ff = y0_ff - 0.5 * g_ff * t_ex_ff**2

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_ex_ff, y_ex_ff, 'k--', lw=2, label='Exact')

for h, col in zip([1.0, 0.5, 0.1], ['tomato', 'steelblue', 'seagreen']):
    t_n, y_n = euler_integrate(freefall_rhs, (0, T_fall*1.05), np.array([y0_ff, v0_ff]), h)
    mask = y_n[:, 0] >= 0
    ax.plot(t_n[mask], y_n[mask, 0], 'o-', color=col, ms=4, lw=1.5, label=f'Euler h={h}')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Height $y$ (m)')
ax.set_title('Free Fall: Euler Method on a 2nd-Order ODE')
ax.legend()
plt.tight_layout()
plt.show()

### Example 5: The Simple Harmonic Oscillator — and an Euler Problem

The **SHO** is the most important system in physics:

$$m\ddot{x} = -kx \implies \frac{dx}{dt} = v, \quad \frac{dv}{dt} = -\omega_0^2 x, \quad \omega_0 = \sqrt{k/m}$$

Exact solution: $x(t) = A\cos(\omega_0 t + \phi)$, with conserved energy $E = \frac{1}{2}mv^2 + \frac{1}{2}kx^2$.

**Energy conservation is a powerful diagnostic:** any deviation from $E = E_0$ is purely numerical error.

The Euler method has a serious flaw here: **it does not conserve energy** — energy *grows* over time, and the oscillation amplitude slowly increases. This is a fundamental property of the method, not a numerical accident.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: SHO phase portrait building step by step (Euler) ─────────────
# Use h=0.2 so the spiraling energy growth is clearly visible
omega0 = 1.0        # angular frequency
x0_sho = 1.0        # initial position (amplitude)
v0_sho = 0.0        # starts at rest
y0_sho = np.array([x0_sho, v0_sho])

def sho_rhs(t, y):
    x, v = y
    return np.array([v, -omega0**2 * x])

def sho_energy(y):
    x, v = y[:, 0], y[:, 1]
    return 0.5 * v**2 + 0.5 * omega0**2 * x**2

def sho_exact(t):
    return x0_sho * np.cos(omega0 * t)

T_sho = 15.0
t_ex_sho = np.linspace(0, T_sho, 200)

h_anim_sho = 0.2
T_anim_sho = T_sho
t_anim_sho = np.arange(0, T_anim_sho + h_anim_sho, h_anim_sho)
y_anim_sho = np.zeros((len(t_anim_sho), 2))
y_anim_sho[0] = y0_sho

theta_exact = np.linspace(0, 2*np.pi, 300)

with LivePlot(figsize=(9, 4)) as lp:
    lp.fig.clf()
    ax_t, ax_ph = lp.fig.subplots(1, 2)
    lp.fig.suptitle(f'SHO: Euler h={h_anim_sho} — watch energy grow', fontsize=11)
    
    ax_t.plot(t_ex_sho, sho_exact(t_ex_sho), 'k--', lw=1.5, alpha=0.4, label='Exact')
    ax_ph.plot(np.cos(theta_exact), np.sin(theta_exact), 'k--', lw=1.5, alpha=0.4, label='Exact circle')
    
    line_t,  = ax_t.plot([], [],  color='tomato', lw=1.5, label=f'Euler h={h_anim_sho}')
    dot_t,   = ax_t.plot([], [],  'o', color='tomato', ms=10, zorder=5)
    line_ph, = ax_ph.plot([], [], color='tomato', lw=1.5)
    dot_ph,  = ax_ph.plot([], [], 'o', color='tomato', ms=10, zorder=5)
    
    ax_t.set_xlim(0, T_anim_sho); ax_t.set_ylim(-3, 3)
    ax_t.set_xlabel('Time (s)'); ax_t.set_ylabel('$x(t)$')
    ax_t.set_title('Position vs time'); ax_t.legend(fontsize=9)
    
    ax_ph.set_xlim(-3, 3); ax_ph.set_ylim(-3, 3)
    ax_ph.set_xlabel('$x$'); ax_ph.set_ylabel('$v$')
    ax_ph.set_title('Phase portrait'); ax_ph.set_aspect('equal')
    ax_ph.legend(fontsize=9)
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_sho)):
        y_anim_sho[i_step] = euler_step(sho_rhs, t_anim_sho[i_step-1], y_anim_sho[i_step-1], h_anim_sho)
        line_t.set_data(t_anim_sho[:i_step+1], y_anim_sho[:i_step+1, 0])
        dot_t.set_data([t_anim_sho[i_step]], [y_anim_sho[i_step, 0]])
        line_ph.set_data(y_anim_sho[:i_step+1, 0], y_anim_sho[:i_step+1, 1])
        dot_ph.set_data([y_anim_sho[i_step, 0]], [y_anim_sho[i_step, 1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: Time-reversed SHO phase portrait (Euler) ─────────────
# Use h=-0.2 so the spiraling energy growth is clearly visible
T_sho = -15.0
t_ex_sho = np.linspace(0, T_sho, 200)

h_anim_sho = -0.2
T_anim_sho = T_sho
t_anim_sho = np.arange(0, T_anim_sho + h_anim_sho, h_anim_sho)
y_anim_sho = np.zeros((len(t_anim_sho), 2))
y_anim_sho[0] = y0_sho

theta_exact = np.linspace(0, 2*np.pi, 300)

with LivePlot(figsize=(9, 4)) as lp:
    lp.fig.clf()
    ax_t, ax_ph = lp.fig.subplots(1, 2)
    lp.fig.suptitle(f'SHO: Euler h={h_anim_sho} — Time Reversed', fontsize=11)
    
    ax_t.plot(t_ex_sho, sho_exact(t_ex_sho), 'k--', lw=1.5, alpha=0.4, label='Exact')
    ax_ph.plot(np.cos(theta_exact), np.sin(theta_exact), 'k--', lw=1.5, alpha=0.4, label='Exact circle')
    
    line_t,  = ax_t.plot([], [],  color='tomato', lw=1.5, label=f'Euler h={h_anim_sho}')
    dot_t,   = ax_t.plot([], [],  'o', color='tomato', ms=10, zorder=5)
    line_ph, = ax_ph.plot([], [], color='tomato', lw=1.5)
    dot_ph,  = ax_ph.plot([], [], 'o', color='tomato', ms=10, zorder=5)
    
    ax_t.set_xlim(T_anim_sho, 0); ax_t.set_ylim(-3, 3)
    ax_t.set_xlabel('Time (s)'); ax_t.set_ylabel('$x(t)$')
    ax_t.set_title('Position vs time'); ax_t.legend(fontsize=9)
    
    ax_ph.set_xlim(-3, 3); ax_ph.set_ylim(-3, 3)
    ax_ph.set_xlabel('$x$'); ax_ph.set_ylabel('$v$')
    ax_ph.set_title('Phase portrait'); ax_ph.set_aspect('equal')
    ax_ph.legend(fontsize=9)
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_sho)):
        y_anim_sho[i_step] = euler_step(sho_rhs, t_anim_sho[i_step-1], y_anim_sho[i_step-1], h_anim_sho)
        line_t.set_data(t_anim_sho[:i_step+1], y_anim_sho[:i_step+1, 0])
        dot_t.set_data([t_anim_sho[i_step]], [y_anim_sho[i_step, 0]])
        line_ph.set_data(y_anim_sho[:i_step+1, 0], y_anim_sho[:i_step+1, 1])
        dot_ph.set_data([y_anim_sho[i_step, 0]], [y_anim_sho[i_step, 1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Simple Harmonic Oscillator with Euler Method
omega0 = 1.0        # angular frequency
x0_sho = 1.0        # initial position (amplitude)
v0_sho = 0.0        # starts at rest
y0_sho = np.array([x0_sho, v0_sho])

def sho_rhs(t, y):
    x, v = y
    return np.array([v, -omega0**2 * x])

def sho_energy(y):
    x, v = y[:, 0], y[:, 1]
    return 0.5 * v**2 + 0.5 * omega0**2 * x**2

def sho_exact(t):
    return x0_sho * np.cos(omega0 * t)

T_sho = 30.0
t_ex_sho = np.linspace(0, T_sho, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_ex_sho, sho_exact(t_ex_sho), 'k--', lw=1.5, alpha=0.5, label='Exact')

for h, col in zip([0.5, 0.2, 0.05], ['tomato', 'steelblue', 'seagreen']):
    t_n, y_n = euler_integrate(sho_rhs, (0, T_sho), y0_sho, h)
    E_n = sho_energy(y_n)
    ax1.plot(t_n, y_n[:, 0], color=col, lw=1.2, label=f'Euler h={h}')
    ax2.plot(t_n, (E_n / E_n[0] - 1) * 100, color=col, lw=1.2, label=f'h={h}')

ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Position $x$')
ax1.set_title('SHO: Euler Method (energy grows!)')
ax1.legend(fontsize=9)

ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Energy error (%)')
ax2.set_title('Relative energy error $(E/E_0 - 1)$')
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 3 — Better Methods: RK2, RK4, and Leapfrog

The Euler method uses only the slope at the *beginning* of each step. We can do much better by being cleverer about which slopes we use.

### A general integrator framework

All the methods we build share the same structure. Let's write a general integrator that takes any step function.

In [ ]:
# General integrator: takes any step function

def integrate(step_fn, f, t_span, y0, h):
    """
    Integrate dy/dt = f(t, y) using the given step_fn.
    step_fn(f, t, y, h) -> y_next
    """
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = step_fn(f, t[i], y[i], h)
    return t, y

print("Generic integrator defined.")

### The Midpoint Method (RK2)

Instead of using the slope at the *start* of the interval, use the slope at the **midpoint** $t_n + h/2$:

$$k_1 = f(t_n,\, y_n) \qquad \text{(Euler estimate to midpoint)}$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2},\, y_n + \tfrac{h}{2}k_1\right) \qquad \text{(slope at midpoint)}$$
$$y_{n+1} = y_n + h\,k_2 \qquad \text{(take the full step using midpoint slope)}$$

This is a **second-order method** ($p=2$): global error $\sim O(h^2)$. Halving $h$ reduces the error by a factor of **4**, at only twice the cost of Euler.

### Heun's Method (another RK2)

Average the slope at the start and the slope at a provisional endpoint:
$$k_1 = f(t_n, y_n), \quad k_2 = f(t_n+h, y_n + h k_1), \quad y_{n+1} = y_n + \frac{h}{2}(k_1 + k_2)$$

Both are second-order; they differ in the leading constant.

In [ ]:
# RK2 methods

def rk2_midpoint_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/2, y + h/2 * k1)
    return y + h * k2

def rk2_heun_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h, y + h * k1)
    return y + h/2 * (k1 + k2)

print("RK2 methods defined.")

### The Fourth-Order Runge-Kutta Method (RK4)

The **classical RK4** uses four slope evaluations to achieve **fourth-order accuracy** ($p=4$):

$$k_1 = f(t_n,\; y_n)$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}k_2\right)$$
$$k_4 = f\!\left(t_n + h,\; y_n + h\,k_3\right)$$
$$y_{n+1} = y_n + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)$$

The weights $1/6, 1/3, 1/3, 1/6$ mirror **Simpson's rule** for numerical integration. Halving $h$ reduces the error by a factor of **16** — this is a dramatic improvement.

RK4 is the workhorse of computational physics. It is the right default method for most problems.

In [ ]:
# Classical RK4

def rk4_step(f, t, y, h):
    k1 = f(t,       y)
    k2 = f(t + h/2, y + h/2 * k1)
    k3 = f(t + h/2, y + h/2 * k2)
    k4 = f(t + h,   y + h   * k3)
    return y + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)

def rk4_integrate(f, t_span, y0, h):
    return integrate(rk4_step, f, t_span, y0, h)

print("RK4 defined.")

### Example 6: Comparing All Methods on the SHO

Let's use a deliberately large step size $h=0.5$ to expose the differences between methods.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: RK4 on the SHO — phase portrait building step by step ────────
# Uses h=0.5 (same as comparison below) so you can see accuracy directly

h_anim_rk4 = 0.5
T_anim_rk4 = 20.0
t_anim_rk4 = np.arange(0, T_anim_rk4 + h_anim_rk4, h_anim_rk4)
y_anim_rk4 = np.zeros((len(t_anim_rk4), 2))
y_anim_rk4[0] = y0_sho
y_anim_eu = np.zeros((len(t_anim_rk4), 2))
y_anim_eu[0] = y0_sho

with LivePlot(figsize=(9, 4)) as lp:
    lp.fig.clf()
    ax_e, ax_r = lp.fig.subplots(1, 2)
    lp.fig.suptitle(f'Euler vs RK4 phase portrait, h={h_anim_rk4}', fontsize=11)
    
    for ax in [ax_e, ax_r]:
        ax.plot(np.cos(theta_exact), np.sin(theta_exact), 'k--', lw=1.5, alpha=0.4)
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
        ax.set_xlabel('$x$'); ax.set_ylabel('$v$'); ax.set_aspect('equal')
    ax_e.set_title('Euler (spirals out)')
    ax_r.set_title('RK4 (stays on circle)')
    
    line_e, = ax_e.plot([], [], color='tomato',    lw=1.5)
    dot_e,  = ax_e.plot([], [], 'o', color='tomato',    ms=10, zorder=5)
    line_r, = ax_r.plot([], [], color='steelblue', lw=1.5)
    dot_r,  = ax_r.plot([], [], 'o', color='steelblue', ms=10, zorder=5)
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_rk4)):
        y_anim_eu[i_step] = euler_step(sho_rhs, t_anim_rk4[i_step-1], y_anim_eu[i_step-1], h_anim_rk4)
        y_anim_rk4[i_step] = rk4_step(sho_rhs, t_anim_rk4[i_step-1], y_anim_rk4[i_step-1], h_anim_rk4)
        line_e.set_data(y_anim_eu[:i_step+1, 0],  y_anim_eu[:i_step+1, 1])
        dot_e.set_data([y_anim_eu[i_step, 0]],    [y_anim_eu[i_step, 1]])
        line_r.set_data(y_anim_rk4[:i_step+1, 0], y_anim_rk4[:i_step+1, 1])
        dot_r.set_data([y_anim_rk4[i_step, 0]],   [y_anim_rk4[i_step, 1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Four-method comparison on the SHO — large step size to show differences
h_cmp  = 0.5
T_cmp  = 20.0
t_ref  = np.linspace(0, T_cmp, 4000)

methods = [
    ('Euler',   euler_step,        'tomato'),
    ('RK2',     rk2_midpoint_step, 'darkorange'),
    ('Heun',    rk2_heun_step,     'goldenrod'),
    ('RK4',     rk4_step,          'steelblue'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, (name, step_fn, col) in zip(axes.flatten(), methods):
    t_m, y_m = integrate(step_fn, sho_rhs, (0, T_cmp), y0_sho, h_cmp)
    E_m = sho_energy(y_m)
    ax.plot(t_ref, sho_exact(t_ref), 'k--', lw=1.5, alpha=0.4, label='Exact')
    ax.plot(t_m, y_m[:, 0], color=col, lw=1.8, label=f'{name}  h={h_cmp}')
    ax2 = ax.twinx()
    ax2.plot(t_m, (E_m/E_m[0] - 1)*100, color=col, alpha=0.4, lw=2, ls=':')
    ax2.set_ylabel('Energy error (%)', color=col, fontsize=9)
    ax2.tick_params(axis='y', colors=col)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('$x(t)$')
    ax.set_title(f'{name}  (h={h_cmp})')
    ax.legend(fontsize=9)

fig.suptitle('All Four Methods on SHO with $h=0.5$', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')
%matplotlib inline

# Error scaling: all four methods
h_vals_err = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]
x_exact_20 = sho_exact(np.array([20.0]))[0]
errors_all = {name: [] for name, _, _ in methods}

for h in h_vals_err:
    for name, step_fn, _ in methods:
        _, y_m = integrate(step_fn, sho_rhs, (0, 20), y0_sho, h)
        errors_all[name].append(abs(y_m[-1, 0] - x_exact_20))

h_arr = np.array(h_vals_err)

fig, ax = plt.subplots(figsize=(8, 5))
for (name, _, col), order in zip(methods, [1, 2, 2, 4]):
    ax.loglog(h_arr, errors_all[name], 'o-', color=col, ms=7, label=f'{name} ($O(h^{order})$)')
ax.loglog(h_arr, 0.6*h_arr,      'k:',  lw=1.5, label='$O(h)$')
ax.loglog(h_arr, 0.08*h_arr**2,  'k--', lw=1.5, label='$O(h^2)$')
ax.loglog(h_arr, 0.01*h_arr**4,  'k-.', lw=1.5, label='$O(h^4)$')
ax.set_xlabel('Step size $h$')
ax.set_ylabel('Global error at $t=20$ s')
ax.set_title('Error Scaling: All Methods on SHO')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### The Symplectic Leapfrog Method

For **Hamiltonian systems** (those with conserved energy), there is a special class of integrators that preserves the geometric structure of phase space. The simplest is the **leapfrog** (or Störmer-Verlet) method:

$$v_{n+1/2} = v_n + \frac{h}{2}\,a(x_n)$$
$$x_{n+1} = x_n + h\,v_{n+1/2}$$
$$v_{n+1} = v_{n+1/2} + \frac{h}{2}\,a(x_{n+1})$$

**Why is this special?** Leapfrog conserves a *shadow Hamiltonian* $\tilde{H} = H + O(h^2)$. Energy oscillates around the true value rather than drifting monotonically. Over millions of steps (molecular dynamics, orbital mechanics), this is crucial.

It is only second-order accurate ($p=2$), but for long-time Hamiltonian dynamics it far outperforms RK4.

In [ ]:
# Symplectic Leapfrog integrator

def leapfrog_integrate(accel_fn, t_span, x0, v0, h):
    """
    accel_fn(x) -> acceleration (no velocity dependence).
    x0, v0 are scalars.
    Returns t, x, v arrays.
    """
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    N = len(t)
    x = np.zeros(N); v = np.zeros(N)
    x[0], v[0] = x0, v0
    for i in range(N - 1):
        v_half = v[i] + h/2 * accel_fn(x[i])
        x[i+1] = x[i] + h * v_half
        v[i+1] = v_half + h/2 * accel_fn(x[i+1])
    return t, x, v

def sho_accel(x):
    return -omega0**2 * x

print("Leapfrog integrator defined.")

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: Watch Euler, RK4, and Leapfrog phase portraits develop ────────
# All three run for 10 time units so differences emerge quickly

h_anim_lt = 0.5
T_anim_lt = 10.0
t_anim_lt = np.arange(0, T_anim_lt + h_anim_lt, h_anim_lt)

# Pre-allocate arrays
y_anim_eu_lt  = np.zeros((len(t_anim_lt), 2)); y_anim_eu_lt[0]  = y0_sho
y_anim_rk4_lt = np.zeros((len(t_anim_lt), 2)); y_anim_rk4_lt[0] = y0_sho
x_anim_lf_lt  = np.zeros(len(t_anim_lt));      x_anim_lf_lt[0]  = x0_sho
v_anim_lf_lt  = np.zeros(len(t_anim_lt));      v_anim_lf_lt[0]  = v0_sho

with LivePlot(figsize=(12, 4)) as lp:
    lp.fig.clf()
    ax_eu, ax_rk, ax_lf = lp.fig.subplots(1, 3)
    lp.fig.suptitle(f'Phase portraits building up (h={h_anim_lt})', fontsize=11)
    
    for ax, title, col in [
        (ax_eu,  'Euler (spirals out)',      'tomato'),
        (ax_rk,  'RK4 (very accurate)',      'goldenrod'),
        (ax_lf,  'Leapfrog (stays bounded)', 'steelblue'),
    ]:
        ax.plot(np.cos(theta_exact), np.sin(theta_exact), 'k--', lw=1.5, alpha=0.3)
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
        ax.set_xlabel('$x$'); ax.set_ylabel('$v$')
        ax.set_title(title); ax.set_aspect('equal')
    
    line_eu, = ax_eu.plot([], [], color='tomato',    lw=1.0)
    dot_eu,  = ax_eu.plot([], [], 'o', color='tomato',    ms=8, zorder=5)
    line_rk, = ax_rk.plot([], [], color='goldenrod', lw=1.0)
    dot_rk,  = ax_rk.plot([], [], 'o', color='goldenrod', ms=8, zorder=5)
    line_lf, = ax_lf.plot([], [], color='steelblue', lw=1.0)
    dot_lf,  = ax_lf.plot([], [], 'o', color='steelblue', ms=8, zorder=5)
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_lt)):
        y_anim_eu_lt[i_step]  = euler_step(sho_rhs, t_anim_lt[i_step-1], y_anim_eu_lt[i_step-1], h_anim_lt)
        y_anim_rk4_lt[i_step] = rk4_step(sho_rhs,  t_anim_lt[i_step-1], y_anim_rk4_lt[i_step-1], h_anim_lt)
        v_half = v_anim_lf_lt[i_step-1] + h_anim_lt/2 * sho_accel(x_anim_lf_lt[i_step-1])
        x_anim_lf_lt[i_step] = x_anim_lf_lt[i_step-1] + h_anim_lt * v_half
        v_anim_lf_lt[i_step] = v_half + h_anim_lt/2 * sho_accel(x_anim_lf_lt[i_step])
        line_eu.set_data(y_anim_eu_lt[:i_step+1, 0],  y_anim_eu_lt[:i_step+1, 1])
        dot_eu.set_data([y_anim_eu_lt[i_step, 0]],    [y_anim_eu_lt[i_step, 1]])
        line_rk.set_data(y_anim_rk4_lt[:i_step+1, 0], y_anim_rk4_lt[:i_step+1, 1])
        dot_rk.set_data([y_anim_rk4_lt[i_step, 0]],   [y_anim_rk4_lt[i_step, 1]])
        line_lf.set_data(x_anim_lf_lt[:i_step+1],     v_anim_lf_lt[:i_step+1])
        dot_lf.set_data([x_anim_lf_lt[i_step]],       [v_anim_lf_lt[i_step]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Long-time comparison: Euler vs RK4 vs Leapfrog on the SHO

T_long = 200.0
h_long = 0.3

t_eu_l,  y_eu_l  = integrate(euler_step, sho_rhs, (0, T_long), y0_sho, h_long)
t_rk4_l, y_rk4_l = rk4_integrate(sho_rhs, (0, T_long), y0_sho, h_long)
t_lf_l,  x_lf_l, v_lf_l = leapfrog_integrate(sho_accel, (0, T_long), x0_sho, v0_sho, h_long)

E_eu_l  = sho_energy(y_eu_l)
E_rk4_l = sho_energy(y_rk4_l)
E_lf_l  = 0.5*v_lf_l**2 + 0.5*omega0**2*x_lf_l**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(t_eu_l,  (E_eu_l/E_eu_l[0]   - 1)*100, 'tomato',    lw=1, label=f'Euler h={h_long}')
ax1.plot(t_rk4_l, (E_rk4_l/E_rk4_l[0] - 1)*100, 'goldenrod', lw=1, label=f'RK4 h={h_long}')
ax1.plot(t_lf_l,  (E_lf_l/E_lf_l[0]   - 1)*100, 'steelblue', lw=0.8, label=f'Leapfrog h={h_long}')
ax1.set_ylim(-10, 10)
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Energy error (%)')
ax1.set_title('Long-time energy: Euler drifts, Leapfrog is bounded')
ax1.legend()

theta = np.linspace(0, 2*np.pi, 300)
ax2.plot(y_eu_l[:, 0],  y_eu_l[:, 1],  'tomato',    lw=0.5, alpha=0.7, label='Euler')
ax2.plot(y_rk4_l[:, 0], y_rk4_l[:, 1], 'goldenrod', lw=0.5, alpha=0.7, label='RK4')
ax2.plot(x_lf_l, v_lf_l, 'steelblue', lw=0.5, alpha=0.7, label='Leapfrog')
ax2.plot(np.cos(theta), np.sin(theta), 'k--', lw=2, label='Exact')
ax2.set_xlim(-5, 5)
ax2.set_ylim(-5, 5)
ax2.set_xlabel('Position $x$'); ax2.set_ylabel('Velocity $v$')
ax2.set_title('Phase portrait: Euler spiral grows, Leapfrog stays on circle')
ax2.legend(fontsize=9); ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

### Method Summary

| Method | Order | f-evals/step | Long-time energy | Notes |
|---|---|---|---|---|
| Euler | 1 | 1 | Drifts upward | Demos only |
| RK2 / Heun | 2 | 2 | Drifts slowly | Quick estimates |
| RK4 | 4 | 4 | Very good | General purpose |
| Leapfrog | 2 | 1 | Bounded (symplectic) | Long Hamiltonian sims |

---
## Part 4 — More Physics: Second-Order Systems

### Example 7: Damped Harmonic Motion

Adding a velocity-dependent friction force $-b\dot{x}$ gives:
$$\ddot{x} = -\omega_0^2 x - 2\gamma\dot{x}, \quad \gamma = b/(2m)$$

Three regimes depending on whether $\gamma < \omega_0$, $\gamma = \omega_0$, or $\gamma > \omega_0$:

| Regime | Condition | Motion |
|---|---|---|
| Underdamped | $\gamma < \omega_0$ | Oscillates with decaying amplitude |
| Critically damped | $\gamma = \omega_0$ | Returns to 0 fastest without oscillating |
| Overdamped | $\gamma > \omega_0$ | Slowly creeps back to 0 |

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: underdamped SHO — watch RK4 build the decaying oscillation ───
def damped_sho_rhs(gamma):
    def rhs(t, y):
        x, v = y
        return np.array([v, -omega0**2 * x - 2*gamma*v])
    return rhs

t_damp = np.linspace(0, 20, 500)
y0_damp = np.array([1.0, 0.0])

gamma_anim = 0.1
h_anim_damp = 0.2
t_anim_damp = np.arange(0, 20 + h_anim_damp, h_anim_damp)
y_anim_damp = np.zeros((len(t_anim_damp), 2))
y_anim_damp[0] = np.array([1.0, 0.0])
rhs_anim_damp = damped_sho_rhs(gamma_anim)

omega1_anim = np.sqrt(omega0**2 - gamma_anim**2)
x_exact_damp = np.exp(-gamma_anim*t_anim_damp) * (
    np.cos(omega1_anim*t_anim_damp) + gamma_anim/omega1_anim*np.sin(omega1_anim*t_anim_damp))

with LivePlot(figsize=(10, 4)) as lp:
    lp.fig.clf()
    ax_xt, ax_ph = lp.fig.subplots(1, 2)
    lp.fig.suptitle(f'Underdamped SHO ($\\gamma={gamma_anim}$): RK4 steps', fontsize=11)
    
    ax_xt.plot(t_anim_damp, x_exact_damp, 'k--', lw=1.5, alpha=0.5, label='Exact')
    line_xt, = ax_xt.plot([], [], color='steelblue', lw=2, label='RK4')
    dot_xt,  = ax_xt.plot([], [], 'o', color='steelblue', ms=10, zorder=5)
    ax_xt.set_xlim(0, 20); ax_xt.set_ylim(-1.1, 1.1)
    ax_xt.set_xlabel('Time (s)'); ax_xt.set_ylabel('$x(t)$')
    ax_xt.set_title('Position vs time'); ax_xt.legend(fontsize=9)
    
    line_ph, = ax_ph.plot([], [], color='steelblue', lw=1.5)
    dot_ph,  = ax_ph.plot([], [], 'o', color='steelblue', ms=10, zorder=5)
    ax_ph.set_xlim(-1.2, 1.2); ax_ph.set_ylim(-1.2, 1.2)
    ax_ph.set_xlabel('$x$'); ax_ph.set_ylabel('$v$')
    ax_ph.set_title('Phase portrait (spiral inward)')
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_damp)):
        y_anim_damp[i_step] = rk4_step(rhs_anim_damp, t_anim_damp[i_step-1], y_anim_damp[i_step-1], h_anim_damp)
        line_xt.set_data(t_anim_damp[:i_step+1], y_anim_damp[:i_step+1, 0])
        dot_xt.set_data([t_anim_damp[i_step]], [y_anim_damp[i_step, 0]])
        line_ph.set_data(y_anim_damp[:i_step+1, 0], y_anim_damp[:i_step+1, 1])
        dot_ph.set_data([y_anim_damp[i_step, 0]], [y_anim_damp[i_step, 1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Damped harmonic oscillator

def damped_sho_rhs(gamma):
    def rhs(t, y):
        x, v = y
        return np.array([v, -omega0**2 * x - 2*gamma*v])
    return rhs

t_damp = np.linspace(0, 20, 500)
y0_damp = np.array([1.0, 0.0])

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

cases = [
    (0.1,  'Underdamped $\\gamma=0.1$',  'steelblue'),
    (1.0,  'Critically damped $\\gamma=1.0$', 'seagreen'),
    (2.0,  'Overdamped $\\gamma=2.0$',   'tomato'),
]

for col_idx, (gamma, label, color) in enumerate(cases):
    t_n, y_n = rk4_integrate(damped_sho_rhs(gamma), (0, 20), y0_damp, 0.05)
    if gamma < omega0:
        omega1 = np.sqrt(omega0**2 - gamma**2)
        x_exact_d = np.exp(-gamma*t_damp) * (np.cos(omega1*t_damp) +
                     gamma/omega1*np.sin(omega1*t_damp))
        axes[0][col_idx].plot(t_damp, x_exact_d, 'k--', lw=1.5, alpha=0.5, label='Exact')
    axes[0][col_idx].plot(t_n, y_n[:, 0], color=color, lw=2, label='RK4')
    axes[0][col_idx].set_xlabel('Time (s)'); axes[0][col_idx].set_ylabel('$x(t)$')
    axes[0][col_idx].set_title(label)
    axes[0][col_idx].legend(fontsize=9)
    axes[1][col_idx].plot(y_n[:, 0], y_n[:, 1], color=color, lw=1.5)
    axes[1][col_idx].set_xlabel('Position $x$'); axes[1][col_idx].set_ylabel('Velocity $v$')
    axes[1][col_idx].set_title('Phase portrait')

fig.suptitle('Damped Harmonic Oscillator — RK4', fontsize=13)
plt.tight_layout()
plt.show()

### Example 8: Energy Decay in the Damped SHO

With damping, energy is no longer conserved — it decays as $E(t) = E_0 e^{-2\gamma t}$. We can verify this numerically.

In [ ]:
plt.close('all')
%matplotlib inline

# Energy decay verification for damped SHO

fig, ax = plt.subplots(figsize=(9, 5))
t_e = np.linspace(0, 20, 500)

for gamma, label, color in [
    (0.05, r'$\gamma=0.05$', 'steelblue'),
    (0.2,  r'$\gamma=0.20$', 'seagreen'),
    (0.5,  r'$\gamma=0.50$', 'tomato'),
]:
    t_n, y_n = rk4_integrate(damped_sho_rhs(gamma), (0, 20), y0_damp, 0.05)
    E_n = 0.5*y_n[:,1]**2 + 0.5*omega0**2*y_n[:,0]**2
    ax.semilogy(t_n, E_n, color=color, lw=2, label=label)
    ax.semilogy(t_e, E_n[0]*np.exp(-2*gamma*t_e), '--', color=color, alpha=0.5)

ax.set_xlabel('Time (s)'); ax.set_ylabel('Energy (log scale)')
ax.set_title('Energy decay: RK4 solution (solid) vs. $E_0 e^{-2\\gamma t}$ (dashed)')
ax.legend()
plt.tight_layout()
plt.show()

### Example 9: The Nonlinear Pendulum

For small angles, a pendulum is an SHO. Without the small-angle approximation:
$$\ddot\theta = -\frac{g}{L}\sin\theta \implies \frac{d\theta}{dt} = \omega, \quad \frac{d\omega}{dt} = -\frac{g}{L}\sin\theta$$

The period now depends on amplitude. The phase portrait reveals two qualitatively different trajectories:
- **Librations** (closed curves): oscillates back and forth
- **Rotations** (open curves): enough energy to go over the top
- **Separatrix**: the boundary between the two regimes, where $E = 2gL$

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: nonlinear pendulum trajectory building step by step ──────────
# Large amplitude (1.5 rad) to make nonlinearity visible
g_pend = 9.81
L_pend = 1.0

def pendulum_rhs(t, y):
    theta, omega = y
    return np.array([omega, -(g_pend/L_pend)*np.sin(theta)])

def pendulum_energy(y):
    theta, omega = y[:, 0], y[:, 1]
    return 0.5*L_pend**2*omega**2 + g_pend*L_pend*(1 - np.cos(theta))

h_anim_pend = 0.05
t_anim_pend = np.arange(0, 5 + h_anim_pend, h_anim_pend)
y_anim_pend = np.zeros((len(t_anim_pend), 2))
y_anim_pend[0] = np.array([1.55, 0.0])

# Small-angle exact for comparison
omega0_pend = np.sqrt(g_pend/L_pend)
x_smallangle = 1.5 * np.cos(omega0_pend * t_anim_pend)

with LivePlot(figsize=(10, 4)) as lp:
    lp.fig.clf()
    ax_xt, ax_ph = lp.fig.subplots(1, 2)
    lp.fig.suptitle(r'Nonlinear pendulum $\theta_0=1.5$ rad: RK4 steps', fontsize=11)
    
    ax_xt.plot(t_anim_pend, np.degrees(x_smallangle), 'k--', lw=1.5, alpha=0.5,
               label='Small-angle approx')
    line_pt, = ax_xt.plot([], [], color='seagreen', lw=2, label='RK4 (exact)')
    dot_pt,  = ax_xt.plot([], [], 'o', color='seagreen', ms=10, zorder=5)
    ax_xt.set_xlim(0, 5); ax_xt.set_ylim(-100, 100)
    ax_xt.set_xlabel('Time (s)')
    ax_xt.set_ylabel(r'$\theta$ (degrees)')
    ax_xt.set_title('Angle vs time')
    ax_xt.legend(fontsize=9, loc="upper right")
    
    line_pp, = ax_ph.plot([], [], color='seagreen', lw=1.5)
    dot_pp,  = ax_ph.plot([], [], 'o', color='seagreen', ms=10, zorder=5)
    ax_ph.set_xlim(-2, 2); ax_ph.set_ylim(-5, 5)
    ax_ph.set_xlabel(r'$\theta$ (rad)')
    ax_ph.set_ylabel(r'$\dot\theta$ (rad/s)')
    ax_ph.set_title('Phase portrait')
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_pend)):
        y_anim_pend[i_step] = rk4_step(pendulum_rhs, t_anim_pend[i_step-1], y_anim_pend[i_step-1], h_anim_pend)
        line_pt.set_data(t_anim_pend[:i_step+1], np.degrees(y_anim_pend[:i_step+1, 0]))
        dot_pt.set_data([t_anim_pend[i_step]], [np.degrees(y_anim_pend[i_step, 0])])
        line_pp.set_data(y_anim_pend[:i_step+1, 0], y_anim_pend[:i_step+1, 1])
        dot_pp.set_data([y_anim_pend[i_step, 0]], [y_anim_pend[i_step, 1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Nonlinear pendulum
g_pend = 9.81
L_pend = 1.0

def pendulum_rhs(t, y):
    theta, omega = y
    return np.array([omega, -(g_pend/L_pend)*np.sin(theta)])

def pendulum_energy(y):
    theta, omega = y[:, 0], y[:, 1]
    return 0.5*L_pend**2*omega**2 + g_pend*L_pend*(1 - np.cos(theta))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 5))

ics_pend = [
    (np.array([0.1,  0.0]), 'steelblue', r'$\theta_0=0.1$ rad (small)'),
    (np.array([1.5,  0.0]), 'seagreen',  r'$\theta_0=1.5$ rad (large)'),
    (np.array([2.9,  0.0]), 'tomato',    r'$\theta_0=2.9$ rad (near sep.)'),
    (np.array([-0.1, 5.0]), 'purple',    r'Rotation ($\omega_0=5$)'),
]

for y0_p, col, lbl in ics_pend:
    t_p, y_p = rk4_integrate(pendulum_rhs, (0, 20), y0_p, 0.01)
    ax1.plot(t_p, np.degrees(y_p[:, 0]), color=col, lw=1.8, label=lbl)
    ax2.plot(y_p[:, 0], y_p[:, 1], color=col, lw=1.5)

ax1.set_xlabel('Time (s)'); ax1.set_ylabel(r'$\theta$ (degrees)')
ax1.set_title('Pendulum: angle vs time'); ax1.legend(fontsize=8)
ax2.set_xlabel(r'$\theta$ (rad)'); ax2.set_ylabel(r'$\dot\theta$ (rad/s)')
ax2.set_title('Phase portrait')

from scipy.special import ellipk
T_small_pend = 2*np.pi*np.sqrt(L_pend/g_pend)
theta0_range = np.linspace(1, 170, 200)
T_exact_pend = [4*np.sqrt(L_pend/g_pend)*ellipk(np.sin(np.radians(th)/2)**2)
                for th in theta0_range]
ax3.plot(theta0_range, T_exact_pend, 'steelblue', lw=2, label='Exact period')
ax3.axhline(T_small_pend, color='tomato', ls='--', lw=2, label=f'Small-angle $T={T_small_pend:.2f}$ s')
ax3.set_xlabel(r'Amplitude $\theta_0$ (degrees)')
ax3.set_ylabel('Period $T$ (s)'); ax3.set_title('Period vs amplitude')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Example 10: Driven Anharmonic Oscillator and Chaos

Adding periodic driving to the damped pendulum:
$$\ddot\theta = -\frac{g}{L}\sin\theta - 2\gamma\dot\theta + A\cos(\Omega t)$$

This system exhibits **deterministic chaos**: two trajectories starting arbitrarily close will diverge exponentially. The Poincaré section (sampling the trajectory once per driving period) reveals a **strange attractor** — a fractal structure in phase space.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: Poincaré section building one dot per drive period ────────────
# Each dot is the state (theta, omega) sampled once per driving period
# Watch the strange attractor emerge from what looks like noise at first
A_drive = 6.0
t_max = 50.0
gamma_d  = 0.5
Omega_d  = 2.0/3.0 * np.sqrt(g_pend/L_pend)
T_drive  = 2*np.pi / Omega_d
h_driven = 0.01

def driven_pendulum_rhs(gamma_d, A_drive, Omega_d):
    def rhs(t, y):
        theta, omega = y
        return np.array([omega, -(g_pend/L_pend)*np.sin(theta) - 2*gamma_d*omega + A_drive*np.cos(Omega_d*t)])
    return rhs

t_anim_driven = np.arange(0, t_max + h_driven, h_driven)
y_anim_driven = np.zeros((len(t_anim_driven), 2))
y_anim_driven[0] = np.array([0.0, 0.0])

rhs_driven = driven_pendulum_rhs(gamma_d, A_drive, Omega_d)

with LivePlot(figsize=(10, 4)) as lp:
    lp.fig.clf()
    ax_xt, ax_ph = lp.fig.subplots(1, 2)
    lp.fig.suptitle(r'Driven pendulum: RK4 steps', fontsize=11)
    
    line_pt, = ax_xt.plot([], [], color='seagreen', lw=2, label='RK4')
    dot_pt,  = ax_xt.plot([], [], 'o', color='seagreen', ms=10, zorder=5)
    ax_xt.set_xlim(0, t_max)
    ax_xt.set_ylim(-1300, 1300)
    ax_xt.set_xlabel('Time (s)')
    ax_xt.set_ylabel(r'$\theta$ (degrees)')
    ax_xt.set_title('Angle vs time')
    ax_xt.legend(fontsize=9, loc="upper right")
    
    line_pp, = ax_ph.plot([], [], color='seagreen', lw=1.5)
    dot_pp,  = ax_ph.plot([], [], 'o', color='seagreen', ms=10, zorder=5)
    ax_ph.set_xlim(-30, 30)
    ax_ph.set_ylim(-20, 20)
    ax_ph.set_xlabel(r'$\theta$ (rad)')
    ax_ph.set_ylabel(r'$\dot\theta$ (rad/s)')
    ax_ph.set_title('Phase portrait')
    lp.fig.tight_layout()
    
    for i_step in range(1, len(t_anim_driven)):
        y_anim_driven[i_step] = rk4_step(rhs_driven, t_anim_driven[i_step-1], y_anim_driven[i_step-1], h_driven)
        if i_step % 50 == 0:
            line_pt.set_data(t_anim_driven[:i_step+1], np.degrees(y_anim_driven[:i_step+1, 0]))
            dot_pt.set_data([t_anim_driven[i_step]], [np.degrees(y_anim_driven[i_step, 0])])
            line_pp.set_data(y_anim_driven[:i_step+1, 0], y_anim_driven[:i_step+1, 1])
            dot_pp.set_data([y_anim_driven[i_step, 0]], [y_anim_driven[i_step, 1]])
            await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Driven damped pendulum: periodic vs chaotic
def driven_pendulum_rhs(gamma_d, A_drive, Omega_d):
    def rhs(t, y):
        theta, omega = y
        return np.array([omega,
                         -(g_pend/L_pend)*np.sin(theta) - 2*gamma_d*omega + A_drive*np.cos(Omega_d*t)])
    return rhs

gamma_d  = 0.5
Omega_d  = 2.0/3.0 * np.sqrt(g_pend/L_pend)
T_drive  = 2*np.pi / Omega_d
h_driven = 0.005

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for col_idx, (A_d, regime) in enumerate([(0.9, 'Periodic'), (30.8, 'Chaotic')]):
    rhs_d = driven_pendulum_rhs(gamma_d, A_d, Omega_d)
    _, y_warmup = rk4_integrate(rhs_d, (0, 200), np.array([0.1, 0.0]), h_driven)
    t_d, y_d = rk4_integrate(rhs_d, (0, 300), y_warmup[-1], h_driven)
    theta_w = np.arctan2(np.sin(y_d[:, 0]), np.cos(y_d[:, 0]))
    axes[0][col_idx].plot(t_d[:4000], theta_w[:4000], 'steelblue', lw=0.6)
    axes[0][col_idx].set_xlabel('Time index'); axes[0][col_idx].set_ylabel(r'$\theta$ (rad)')
    axes[0][col_idx].set_title(f'{regime} regime ($A={A_d}$)')
    axes[1][col_idx].scatter(theta_w, y_d[:, 1], s=0.2, color='steelblue', alpha=0.3)
    axes[1][col_idx].set_xlabel(r'$\theta$'); axes[1][col_idx].set_ylabel(r'$\dot\theta$')
    axes[1][col_idx].set_title(f'Phase portrait ({regime})')

rhs_chaos = driven_pendulum_rhs(gamma_d, 30.8, Omega_d)
_, y_wc = rk4_integrate(rhs_chaos, (0, 200), np.array([0.1, 0.0]), h_driven)
t_long, y_long = rk4_integrate(rhs_chaos, (0, 2000), y_wc[-1], h_driven)
spp = int(round(T_drive / h_driven))
pidx = np.arange(0, len(t_long), spp)
axes[0][2].scatter(
    np.arctan2(np.sin(y_long[pidx, 0]), np.cos(y_long[pidx, 0])),
    y_long[pidx, 1], s=1, color='tomato', alpha=0.5)
axes[0][2].set_xlabel(r'$\theta$'); axes[0][2].set_ylabel(r'$\dot\theta$')
axes[0][2].set_title('Poincaré section (chaotic)')

delta0 = 1e-4
y0a = y_wc[-1].copy(); y0b = y_wc[-1].copy() + np.array([delta0, 0.0])
t_bf, ya = rk4_integrate(rhs_chaos, (0, 60), y0a, h_driven)
_,    yb = rk4_integrate(rhs_chaos, (0, 60), y0b, h_driven)
sep = np.sqrt(np.sum((ya - yb)**2, axis=1))
sep[sep < 1e-15] = 1e-15
#axes[1][2].semilogy(t_bf, sep, 'purple', lw=1.5)
axes[1][2].plot(t_bf, ya[:,0], 'purple', lw=1.5)
axes[1][2].plot(t_bf, yb[:,0], 'green', lw=1.5)
axes[1][2].set_xlabel('Time (s)'); axes[1][2].set_ylabel(r'$|\delta|$')
axes[1][2].set_title('Butterfly effect: trajectory separation')

plt.tight_layout()
plt.show()

---
## Part 5 — Two-Dimensional ODE Systems

Extending to two spatial dimensions is conceptually straightforward: the state vector gains more components.

For a particle in the $xy$-plane:
$$\frac{d}{dt}\begin{pmatrix} x \\ y \\ v_x \\ v_y \end{pmatrix} = \begin{pmatrix} v_x \\ v_y \\ F_x/m \\ F_y/m \end{pmatrix}$$

All the integrators we have built work identically — `y` just has four components instead of two.

### Example 11: Projectile Motion

A ball launched at speed $v_0$ and angle $\theta$ under gravity alone.

Exact solution: $x(t) = v_0\cos\theta\cdot t$, $y(t) = v_0\sin\theta\cdot t - \frac{1}{2}gt^2$.

This provides a simple check: RK4 should match the analytic trajectory to near floating-point precision.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: projectile at 45° — RK4 steps tracing the arc ────────────────
g_proj = 9.81

def projectile_rhs_2d(t, state):
    x, y, vx, vy = state
    return np.array([vx, vy, 0.0, -g_proj])

v0_proj = 30.0  # m/s

theta_45 = np.radians(45)
vx0_45 = v0_proj*np.cos(theta_45); vy0_45 = v0_proj*np.sin(theta_45)
T_fl_45 = 2*vy0_45/g_proj
state0_45 = np.array([0.0, 0.0, vx0_45, vy0_45])

h_anim_proj = T_fl_45 / 30   # ~30 visible steps
t_anim_proj = np.arange(0, T_fl_45 * 1.05, h_anim_proj)
y_anim_proj = np.zeros((len(t_anim_proj), 4))
y_anim_proj[0] = state0_45

# Exact arc for reference
t_ex_45 = np.linspace(0, T_fl_45, 300)
x_ex_45 = vx0_45 * t_ex_45
y_ex_45 = vy0_45 * t_ex_45 - 0.5*g_proj*t_ex_45**2

with LivePlot(figsize=(9, 5)) as lp:
    lp.ax.plot(x_ex_45, y_ex_45, 'k--', lw=2, alpha=0.5, label='Exact')
    lp.ax.axhline(0, color='saddlebrown', lw=2, alpha=0.4)
    line_pr, = lp.ax.plot([], [], 'o-', color='steelblue', lw=2, ms=8, label='RK4 steps')
    # Velocity arrow
    arr = lp.ax.annotate('', xy=(0,0), xytext=(0,0),
                          arrowprops=dict(arrowstyle='->', color='tomato', lw=2))
    lp.ax.set_xlim(-5, x_ex_45[-1] * 1.1)
    lp.ax.set_ylim(-5, max(y_ex_45) * 1.2)
    lp.ax.set_xlabel('$x$ (m)'); lp.ax.set_ylabel('$y$ (m)')
    lp.ax.set_title(f'Projectile 45°: RK4 steps (h={h_anim_proj:.2f} s)')
    lp.ax.legend()
    
    for i_step in range(1, len(t_anim_proj)):
        y_anim_proj[i_step] = rk4_step(projectile_rhs_2d, t_anim_proj[i_step-1], y_anim_proj[i_step-1], h_anim_proj)
        if y_anim_proj[i_step, 1] < -0.5:
            break
        line_pr.set_data(y_anim_proj[:i_step+1, 0], y_anim_proj[:i_step+1, 1])
        # Update velocity arrow
        x_cur = y_anim_proj[i_step, 0]; y_cur = y_anim_proj[i_step, 1]
        vx_cur = y_anim_proj[i_step, 2]; vy_cur = y_anim_proj[i_step, 3]
        scale = 0.3
        arr.set_position((x_cur, y_cur))
        arr.xy = (x_cur + scale*vx_cur, y_cur + scale*vy_cur)
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# 2-D projectile motion
g_proj = 9.81

def projectile_rhs_2d(t, state):
    x, y, vx, vy = state
    return np.array([vx, vy, 0.0, -g_proj])

v0_proj = 30.0  # m/s

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors_ang = ['steelblue', 'seagreen', 'tomato', 'darkorange']
for angle_deg, col in zip([30, 45, 60, 75], colors_ang):
    theta_r = np.radians(angle_deg)
    vx0, vy0 = v0_proj*np.cos(theta_r), v0_proj*np.sin(theta_r)
    T_flight = 2*vy0/g_proj
    state0 = np.array([0.0, 0.0, vx0, vy0])
    t_p, y_p = rk4_integrate(projectile_rhs_2d, (0, T_flight*1.02), state0, T_flight/500)
    mask = y_p[:, 1] >= -0.05
    ax1.plot(y_p[mask, 0], y_p[mask, 1], color=col, lw=2, label=f'{angle_deg}°')
    t_ex = np.linspace(0, T_flight, 300)
    ax1.plot(vx0*t_ex, vy0*t_ex - 0.5*g_proj*t_ex**2, 'k--', lw=1, alpha=0.3)

h_vals_proj = [0.2, 0.1, 0.05, 0.02, 0.01, 0.005]
err_proj = []
theta_r = np.radians(45); vx0 = v0_proj*np.cos(theta_r); vy0 = v0_proj*np.sin(theta_r)
T_fl = 2*vy0/g_proj; state0 = np.array([0.0, 0.0, vx0, vy0])
for h in h_vals_proj:
    _, y_h = rk4_integrate(projectile_rhs_2d, (0, T_fl), state0, h)
    x_exact_end = vx0 * T_fl
    err_proj.append(abs(y_h[-1, 0] - x_exact_end))
ax2.loglog(h_vals_proj, err_proj, 'o-', color='steelblue', ms=8, label='RK4 error')
h_arr_p = np.array(h_vals_proj)
ax2.loglog(h_arr_p, 0.01*h_arr_p**4, 'k--', label='$O(h^4)$')
ax2.set_xlabel('Step size $h$'); ax2.set_ylabel('Error in range (m)')
ax2.set_title('RK4 error scaling on projectile'); ax2.legend()

ax1.set_xlabel('$x$ (m)'); ax1.set_ylabel('$y$ (m)')
ax1.set_title(f'Projectile trajectories ($v_0={v0_proj}$ m/s)')
ax1.legend(fontsize=9); ax1.set_ylim(bottom=-0.5)
plt.tight_layout()
plt.show()

### Example 12: Projectile with Air Resistance

Real projectiles experience drag. Two models:

**Linear drag** (valid at low speed): $\mathbf{F}_{\rm drag} = -b\mathbf{v}$

**Quadratic drag** (valid at high speed): $\mathbf{F}_{\rm drag} = -c|\mathbf{v}|\mathbf{v}$

With quadratic drag there is **no closed-form solution** — numerical integration is essential.

In [ ]:
plt.close('all')
%matplotlib inline

# Projectile with drag

def projectile_quad_drag(c_coeff, m=1.0):
    def rhs(t, state):
        x, y, vx, vy = state
        speed = np.sqrt(vx**2 + vy**2)
        return np.array([vx, vy,
                         -(c_coeff/m)*speed*vx,
                         -g_proj - (c_coeff/m)*speed*vy])
    return rhs

v0_d = 50.0; theta_d = np.radians(45)
vx0_d = v0_d*np.cos(theta_d); vy0_d = v0_d*np.sin(theta_d)
state0_d = np.array([0.0, 0.0, vx0_d, vy0_d])
T_nd = 2*vy0_d/g_proj

drag_cases = [
    ('No drag',       lambda t,s: np.array([s[2],s[3],0.0,-g_proj]), 'steelblue'),
    ('Quad $c=0.01$', projectile_quad_drag(0.01), 'seagreen'),
    ('Quad $c=0.05$', projectile_quad_drag(0.05), 'tomato'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for name, rhs_fn, col in drag_cases:
    t_d, y_d = rk4_integrate(rhs_fn, (0, T_nd*2), state0_d, T_nd/500)
    mask_d = y_d[:, 1] >= -0.1
    ax1.plot(y_d[mask_d, 0], y_d[mask_d, 1], color=col, lw=2, label=name)
    E_d = 0.5*(y_d[:, 2]**2 + y_d[:, 3]**2) + g_proj*y_d[:, 1]
    ax2.plot(t_d, E_d, color=col, lw=2, label=name)

ax1.set_xlabel('$x$ (m)'); ax1.set_ylabel('$y$ (m)')
ax1.set_title(f'Projectile with drag ($v_0={v0_d}$ m/s, $45°$)')
ax1.legend(fontsize=9); ax1.set_ylim(bottom=-1)

ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Mechanical energy (J/kg)')
ax2.set_title('Energy dissipated by drag')
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Example 13: Optimal Launch Angle with Drag

Without drag, the optimal launch angle for maximum range is exactly 45°. With quadratic drag, the answer is different. Let's find it numerically.

In [ ]:
plt.close('all')
%matplotlib inline

# Optimal launch angle vs drag coefficient

def compute_range(angle_deg, c_coeff, v0=30.0):
    """Compute range for given launch angle and drag."""
    theta_r = np.radians(angle_deg)
    vx0 = v0*np.cos(theta_r); vy0 = v0*np.sin(theta_r)
    T_max = 2*vy0/g_proj * 3.0
    if c_coeff == 0:
        rhs_fn = lambda t, s: np.array([s[2], s[3], 0.0, -g_proj])
    else:
        rhs_fn = projectile_quad_drag(c_coeff)
    _, y_r = rk4_integrate(rhs_fn, (0, T_max), np.array([0.0, 0.0, vx0, vy0]), T_max/500)
    ground = np.where(y_r[:, 1] < -0.1)[0]
    return y_r[ground[0]-1, 0] if len(ground) > 0 else y_r[-1, 0]

angles_r = np.linspace(5, 85, 50)
drag_coeffs = [0.0, 0.01, 0.05, 0.15]
colors_drag = ['steelblue', 'seagreen', 'tomato', 'purple']

fig, ax = plt.subplots(figsize=(10, 5))
for c, col in zip(drag_coeffs, colors_drag):
    ranges = [compute_range(a, c) for a in angles_r]
    opt_angle = angles_r[np.argmax(ranges)]
    label = f'$c={c}$  (opt={opt_angle:.1f}°)'
    ax.plot(angles_r, ranges, color=col, lw=2, label=label)

ax.axvline(45, color='gray', ls='--', lw=1, alpha=0.5, label='45° (no drag optimum)')
ax.set_xlabel('Launch angle (degrees)'); ax.set_ylabel('Range (m)')
ax.set_title('Range vs Launch Angle: Effect of Quadratic Drag ($v_0=30$ m/s)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Example 14: Planetary Orbits

Newton's law of gravitation:
$$\ddot{x} = -\frac{GM\,x}{r^3}, \quad \ddot{y} = -\frac{GM\,y}{r^3}, \quad r = \sqrt{x^2+y^2}$$

State vector: $(x, y, v_x, v_y)$. Conservation laws: energy $E$ and angular momentum $L$.

We use natural units $GM=1$ so a circular orbit at $r=1$ has period $T=2\pi$.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: elliptical orbit (e=0.8) building step by step ───────────────
# Shows how the planet speeds up near perihelion and slows near aphelion
GM = 1.0

def gravity_rhs(t, state):
    x, y, vx, vy = state
    r3 = (x**2 + y**2)**1.5
    return np.array([vx, vy, -GM*x/r3, -GM*y/r3])

def orbital_energy(state):
    x, y, vx, vy = state[:, 0], state[:, 1], state[:, 2], state[:, 3]
    return 0.5*(vx**2 + vy**2) - GM/np.sqrt(x**2 + y**2)

def angular_momentum(state):
    x, y, vx, vy = state[:, 0], state[:, 1], state[:, 2], state[:, 3]
    return x*vy - y*vx

eccentricities = [0.0, 0.5, 0.8, 0.95]
colors_orb = ['steelblue', 'seagreen', 'tomato', 'darkorange']

e_anim = 0.8
r_p_anim = 1.0
v_p_anim = np.sqrt(GM*(1+e_anim)/r_p_anim)
a_anim = r_p_anim/(1-e_anim)
T_orb_anim = 2*np.pi*np.sqrt(a_anim**3/GM)
h_orb_anim = T_orb_anim / 80
state0_orb_anim = np.array([r_p_anim, 0.0, 0.0, v_p_anim])

t_orb_a = np.arange(0, 2*T_orb_anim + h_orb_anim, h_orb_anim)
y_orb_a = np.zeros((len(t_orb_a), 4))
y_orb_a[0] = state0_orb_anim
for i_step in range(1, len(t_orb_a)):
    y_orb_a[i_step] = rk4_step(gravity_rhs, t_orb_a[i_step-1], y_orb_a[i_step-1], h_orb_anim)

# Velocity magnitude for color-coding speed
speeds = np.sqrt(y_orb_a[:, 2]**2 + y_orb_a[:, 3]**2)

with LivePlot(figsize=(7, 7)) as lp:
    lp.ax.plot(0, 0, 'y*', ms=18, zorder=10, label='Star')
    trail, = lp.ax.plot([], [], color='tomato', lw=1.5, alpha=0.5, label=f'$e={e_anim}$ orbit')
    planet, = lp.ax.plot([], [], 'o', color='tomato', ms=12, zorder=5)
    vel_text = lp.ax.text(0.02, 0.97, '', transform=lp.ax.transAxes, va='top', fontsize=10)
    lp.ax.set_xlim(-8.5, 1.5)
    lp.ax.set_ylim(-3.0, 3.0)
    lp.ax.set_xlabel('$x$ (AU)')
    lp.ax.set_ylabel('$y$ (AU)')
    lp.ax.set_title(f'Elliptical orbit $e={e_anim}$: watch speed vary')
    lp.ax.set_aspect('equal')
    lp.ax.legend(fontsize=9)
    
    for i_step in range(1, len(t_orb_a)):
        trail.set_data(y_orb_a[:i_step+1, 0], y_orb_a[:i_step+1, 1])
        planet.set_data([y_orb_a[i_step, 0]], [y_orb_a[i_step, 1]])
        vel_text.set_text(f'Speed: {speeds[i_step]:.3f} AU/yr')
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Gravitational two-body problem

GM = 1.0

def gravity_rhs(t, state):
    x, y, vx, vy = state
    r3 = (x**2 + y**2)**1.5
    return np.array([vx, vy, -GM*x/r3, -GM*y/r3])

def orbital_energy(state):
    x, y, vx, vy = state[:, 0], state[:, 1], state[:, 2], state[:, 3]
    return 0.5*(vx**2 + vy**2) - GM/np.sqrt(x**2 + y**2)

def angular_momentum(state):
    x, y, vx, vy = state[:, 0], state[:, 1], state[:, 2], state[:, 3]
    return x*vy - y*vx

eccentricities = [0.0, 0.5, 0.8, 0.95]
colors_orb = ['steelblue', 'seagreen', 'tomato', 'darkorange']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 5))

for e, col in zip(eccentricities, colors_orb):
    r_peri = 1.0
    v_peri = np.sqrt(GM*(1+e)/r_peri)
    a = r_peri/(1-e)
    T_orb = 2*np.pi*np.sqrt(a**3/GM)
    state0_orb = np.array([r_peri, 0.0, 0.0, v_peri])
    t_orb, y_orb = rk4_integrate(gravity_rhs, (0, 2*T_orb), state0_orb, T_orb/2000)
    E_orb = orbital_energy(y_orb)
    L_orb = angular_momentum(y_orb)
    label = f'$e={e}$'
    ax1.plot(y_orb[:, 0], y_orb[:, 1], color=col, lw=1.8, label=label)
    ax2.semilogy(t_orb, np.abs(E_orb/E_orb[0]-1)+1e-16, color=col, lw=1.2, label=label)
    ax3.semilogy(t_orb, np.abs(L_orb/L_orb[0]-1)+1e-16, color=col, lw=1.2, label=label)

ax1.plot(0, 0, 'y*', ms=15, zorder=5, label='Star')
ax1.set_xlabel('$x$'); ax1.set_ylabel('$y$')
ax1.set_title('Keplerian orbits (RK4)'); ax1.legend(fontsize=9); ax1.set_aspect('equal')
ax2.set_xlabel('Time'); ax2.set_ylabel('|Energy error|')
ax2.set_title('Energy conservation'); ax2.legend(fontsize=9)
ax3.set_xlabel('Time'); ax3.set_ylabel('|Ang. mom. error|')
ax3.set_title('Angular momentum conservation'); ax3.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Example 15: Long-Time Orbits — Leapfrog vs RK4

For long-time orbital integration, the symplectic leapfrog method is far superior to RK4.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: binary orbit using leapfrog — both stars tracing their paths ─
G_bin = 1.0

def binary_rhs(M1, M2):
    def rhs(t, state):
        x1, y1, x2, y2, vx1, vy1, vx2, vy2 = state
        dx, dy = x2-x1, y2-y1
        r3 = (dx**2 + dy**2)**1.5
        ax1 =  G_bin*M2*dx/r3; ay1 =  G_bin*M2*dy/r3
        ax2 = -G_bin*M1*dx/r3; ay2 = -G_bin*M1*dy/r3
        return np.array([vx1, vy1, vx2, vy2, ax1, ay1, ax2, ay2])
    return rhs

def binary_energy(state, M1, M2):
    x1,y1,x2,y2 = state[:,0],state[:,1],state[:,2],state[:,3]
    vx1,vy1,vx2,vy2 = state[:,4],state[:,5],state[:,6],state[:,7]
    r = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    return 0.5*M1*(vx1**2+vy1**2) + 0.5*M2*(vx2**2+vy2**2) - G_bin*M1*M2/r

# Equal-mass binary in circular orbits
M1b, M2b = 1.0, 1.0
r_sep = 2.0
v_c_bin = np.sqrt(G_bin*M2b/r_sep)
state0_bin = np.array([-r_sep/2, 0, r_sep/2, 0, 0, 0.707*v_c_bin, 0, -0.707*v_c_bin])
T_bin = 2*np.pi*(r_sep/2)/v_c_bin
rhs_bin = binary_rhs(M1b, M2b)
t_bin, y_bin = rk4_integrate(rhs_bin, (0, 10*T_bin), state0_bin, T_bin/150)
E_bin = binary_energy(y_bin, M1b, M2b)

# Equal-mass binary, animate one full orbit
h_anim_bin = T_bin / 120
t_anim_bin = np.arange(0, T_bin + h_anim_bin, h_anim_bin)
y_anim_bin = np.zeros((len(t_anim_bin), 8))
y_anim_bin[0] = state0_bin.copy()
for i_step in range(1, len(t_anim_bin)):
    y_anim_bin[i_step] = rk4_step(rhs_bin, t_anim_bin[i_step-1], y_anim_bin[i_step-1], h_anim_bin)

with LivePlot(figsize=(7, 7)) as lp:
    lp.ax.set_xlim(-1.5, 1.5)
    lp.ax.set_ylim(-1.5, 1.5)
    lp.ax.set_xlabel('$x$'); lp.ax.set_ylabel('$y$')
    lp.ax.set_title('Equal-mass binary: RK4 steps')
    lp.ax.set_aspect('equal')
    
    trail1, = lp.ax.plot([], [], color='steelblue', lw=1.5, alpha=0.5)
    trail2, = lp.ax.plot([], [], color='tomato',    lw=1.5, alpha=0.5)
    star1,  = lp.ax.plot([], [], 'o', color='steelblue', ms=14, zorder=5, label='Star 1')
    star2,  = lp.ax.plot([], [], 'o', color='tomato',    ms=14, zorder=5, label='Star 2')
    com,    = lp.ax.plot([], [], '+', color='k', ms=12, zorder=6, label='CoM')
    lp.ax.legend(fontsize=9)
    
    for i_step in range(1, len(t_anim_bin)):
        trail1.set_data(y_anim_bin[:i_step+1, 0], y_anim_bin[:i_step+1, 1])
        trail2.set_data(y_anim_bin[:i_step+1, 2], y_anim_bin[:i_step+1, 3])
        star1.set_data([y_anim_bin[i_step, 0]], [y_anim_bin[i_step, 1]])
        star2.set_data([y_anim_bin[i_step, 2]], [y_anim_bin[i_step, 3]])
        xcm_s = (M1b*y_anim_bin[i_step,0] + M2b*y_anim_bin[i_step,2])/(M1b+M2b)
        ycm_s = (M1b*y_anim_bin[i_step,1] + M2b*y_anim_bin[i_step,3])/(M1b+M2b)
        com.set_data([xcm_s], [ycm_s])
        await lp.update()

In [ ]:
plt.close('all')
%matplotlib inline

# Leapfrog for 2-D gravity
def grav_accel_2d(pos):
    r3 = (pos[0]**2 + pos[1]**2)**1.5
    return np.array([-GM*pos[0]/r3, -GM*pos[1]/r3])

def leapfrog_2d(accel_fn, t_span, x0, v0, h):
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    N = len(t)
    x = np.zeros((N, 2)); v = np.zeros((N, 2))
    x[0], v[0] = x0, v0
    for i in range(N - 1):
        v_half = v[i] + h/2 * accel_fn(x[i])
        x[i+1] = x[i] + h * v_half
        v[i+1] = v_half + h/2 * accel_fn(x[i+1])
    return t, x, v

e_test = 0.3; r_p = 1.0; v_p = np.sqrt(GM*(1+e_test)/r_p)
state0_lt = np.array([r_p, 0.0, 0.0, v_p])
a_lt = r_p/(1-e_test); T_lt = 2*np.pi*np.sqrt(a_lt**3/GM)
T_sim_lt = 100*T_lt; h_lt = T_lt/100

t_rk4_lt, y_rk4_lt = rk4_integrate(gravity_rhs, (0, T_sim_lt), state0_lt, h_lt)
t_lf_lt, x_lf_lt, v_lf_lt = leapfrog_2d(grav_accel_2d, (0, T_sim_lt),
                                           np.array([r_p, 0.0]), np.array([0.0, v_p]), h_lt)

E_rk4_lt = orbital_energy(y_rk4_lt)
E_lf_lt  = 0.5*(v_lf_lt[:,0]**2 + v_lf_lt[:,1]**2) - GM/np.sqrt(x_lf_lt[:,0]**2 + x_lf_lt[:,1]**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(t_rk4_lt/T_lt, (E_rk4_lt/E_rk4_lt[0]-1)*100, 'goldenrod', lw=0.7, label='RK4 (secular drift)')
ax1.plot(t_lf_lt/T_lt,  (E_lf_lt/E_lf_lt[0]-1)*100,   'steelblue', lw=0.7, label='Leapfrog (bounded)')
ax1.set_xlabel('Orbits'); ax1.set_ylabel('Energy error (%)')
ax1.set_title('100-orbit energy drift (h=T/100)'); ax1.legend()

n_last = int(10*T_lt/h_lt)
ax2.plot(y_rk4_lt[-n_last:, 0], y_rk4_lt[-n_last:, 1], 'goldenrod', lw=0.8, label='RK4')
ax2.plot(x_lf_lt[-n_last:, 0],  x_lf_lt[-n_last:, 1],  'steelblue', lw=0.8, label='Leapfrog')
ax2.plot(0, 0, 'y*', ms=12, zorder=5)
ax2.set_xlabel('$x$'); ax2.set_ylabel('$y$')
ax2.set_title('Last 10 orbits: orbital shape drift')
ax2.legend(); ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## Part 6 — Multi-Body Problems

### Binary Star System

When two bodies have comparable masses, both orbit their common center of mass. The state vector is $\mathbf{y} = (x_1, y_1, x_2, y_2, v_{x1}, v_{y1}, v_{x2}, v_{y2}) \in \mathbb{R}^8$.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: unequal-mass binary — both stars orbit their common CoM ───────
G_bin = 1.0

def binary_rhs(M1, M2):
    def rhs(t, state):
        x1, y1, x2, y2, vx1, vy1, vx2, vy2 = state
        dx, dy = x2-x1, y2-y1
        r3 = (dx**2 + dy**2)**1.5
        ax1 =  G_bin*M2*dx/r3; ay1 =  G_bin*M2*dy/r3
        ax2 = -G_bin*M1*dx/r3; ay2 = -G_bin*M1*dy/r3
        return np.array([vx1, vy1, vx2, vy2, ax1, ay1, ax2, ay2])
    return rhs

def binary_energy(state, M1, M2):
    x1,y1,x2,y2 = state[:,0],state[:,1],state[:,2],state[:,3]
    vx1,vy1,vx2,vy2 = state[:,4],state[:,5],state[:,6],state[:,7]
    r = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    return 0.5*M1*(vx1**2+vy1**2) + 0.5*M2*(vx2**2+vy2**2) - G_bin*M1*M2/r

# Unequal-mass binary
M1u, M2u = 3.0, 1.0
r_u = 2.0
x1_u = -M2u/(M1u+M2u)*r_u
x2_u = M1u/(M1u+M2u)*r_u
v_orb_u = np.sqrt(G_bin*(M1u+M2u)/r_u)
vy1_u =  M2u/(M1u+M2u)*v_orb_u
vy2_u = -M1u/(M1u+M2u)*v_orb_u
state0_u = np.array([x1_u, 0, x2_u, 0, 0, vy1_u, 0, vy2_u])
T_u = 2*np.pi*r_u/v_orb_u
rhs_u = binary_rhs(M1u, M2u)
t_u, y_u = rk4_integrate(rhs_u, (0, 6*T_u), state0_u, T_u/400)
E_u = binary_energy(y_u, M1u, M2u)

h_anim_u = T_u / 120
t_anim_u = np.arange(0, T_u + h_anim_u, h_anim_u)
y_anim_u = np.zeros((len(t_anim_u), 8))
y_anim_u[0] = state0_u.copy()
for i_step in range(1, len(t_anim_u)):
    y_anim_u[i_step] = rk4_step(rhs_u, t_anim_u[i_step-1], y_anim_u[i_step-1], h_anim_u)

with LivePlot(figsize=(7, 7)) as lp:
    lp.ax.set_xlim(-2.5, 2.5); lp.ax.set_ylim(-2.5, 2.5)
    lp.ax.set_xlabel('$x$'); lp.ax.set_ylabel('$y$')
    lp.ax.set_title(f'Unequal binary ($M_1={M1u}$, $M_2={M2u}$): RK4 steps')
    lp.ax.set_aspect('equal')
    
    trail1u, = lp.ax.plot([], [], color='steelblue', lw=1.5, alpha=0.5)
    trail2u, = lp.ax.plot([], [], color='tomato',    lw=1.5, alpha=0.5)
    s1u, = lp.ax.plot([], [], 'o', color='steelblue', ms=int(M1u*6), zorder=5,
                      label=f'Star 1 ($M={M1u}$)')
    s2u, = lp.ax.plot([], [], 'o', color='tomato',    ms=int(M2u*6)*2, zorder=5,
                      label=f'Star 2 ($M={M2u}$)')
    comu, = lp.ax.plot([], [], '+k', ms=14, zorder=6, label='CoM')
    lp.ax.legend(fontsize=9)
    
    for i_step in range(1, len(t_anim_u)):
        trail1u.set_data(y_anim_u[:i_step+1, 0], y_anim_u[:i_step+1, 1])
        trail2u.set_data(y_anim_u[:i_step+1, 2], y_anim_u[:i_step+1, 3])
        s1u.set_data([y_anim_u[i_step, 0]], [y_anim_u[i_step, 1]])
        s2u.set_data([y_anim_u[i_step, 2]], [y_anim_u[i_step, 3]])
        xcm_u = (M1u*y_anim_u[i_step,0]+M2u*y_anim_u[i_step,2])/(M1u+M2u)
        ycm_u = (M1u*y_anim_u[i_step,1]+M2u*y_anim_u[i_step,3])/(M1u+M2u)
        comu.set_data([xcm_u], [ycm_u])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Binary star system
G_bin = 1.0

def binary_rhs(M1, M2):
    def rhs(t, state):
        x1, y1, x2, y2, vx1, vy1, vx2, vy2 = state
        dx, dy = x2-x1, y2-y1
        r3 = (dx**2 + dy**2)**1.5
        ax1 =  G_bin*M2*dx/r3; ay1 =  G_bin*M2*dy/r3
        ax2 = -G_bin*M1*dx/r3; ay2 = -G_bin*M1*dy/r3
        return np.array([vx1, vy1, vx2, vy2, ax1, ay1, ax2, ay2])
    return rhs

def binary_energy(state, M1, M2):
    x1,y1,x2,y2 = state[:,0],state[:,1],state[:,2],state[:,3]
    vx1,vy1,vx2,vy2 = state[:,4],state[:,5],state[:,6],state[:,7]
    r = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    return 0.5*M1*(vx1**2+vy1**2) + 0.5*M2*(vx2**2+vy2**2) - G_bin*M1*M2/r

# Equal-mass binary in circular orbits
M1b, M2b = 1.0, 1.0
r_sep = 2.0
v_c_bin = np.sqrt(G_bin*M2b/r_sep)
state0_bin = np.array([-r_sep/2, 0, r_sep/2, 0, 0, 0.707*v_c_bin, 0, -0.707*v_c_bin])
T_bin = 2*np.pi*(r_sep/2)/v_c_bin
rhs_bin = binary_rhs(M1b, M2b)
t_bin, y_bin = rk4_integrate(rhs_bin, (0, 5*T_bin), state0_bin, T_bin/300)
E_bin = binary_energy(y_bin, M1b, M2b)

# Unequal-mass binary
M1u, M2u = 3.0, 1.0; r_u = 2.0
x1_u = -M2u/(M1u+M2u)*r_u
x2_u = M1u/(M1u+M2u)*r_u
v_orb_u = np.sqrt(G_bin*(M1u+M2u)/r_u)
vy1_u =  M2u/(M1u+M2u)*v_orb_u
vy2_u = -M1u/(M1u+M2u)*v_orb_u
state0_u = np.array([x1_u, 0, x2_u, 0, 0, vy1_u, 0, vy2_u])
T_u = 2*np.pi*r_u/v_orb_u
rhs_u = binary_rhs(M1u, M2u)
t_u, y_u = rk4_integrate(rhs_u, (0, 6*T_u), state0_u, T_u/400)
E_u = binary_energy(y_u, M1u, M2u)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 5))

ax1.plot(y_bin[:,0], y_bin[:,1], 'steelblue', lw=1.5, label='Star 1')
ax1.plot(y_bin[:,2], y_bin[:,3], 'tomato',    lw=1.5, label='Star 2')
xcm = (M1b*y_bin[:,0] + M2b*y_bin[:,2])/(M1b+M2b)
ycm = (M1b*y_bin[:,1] + M2b*y_bin[:,3])/(M1b+M2b)
ax1.plot(xcm, ycm, 'k--', lw=1, alpha=0.4, label='CoM')
ax1.set_title('Equal-mass binary'); ax1.legend(fontsize=9); ax1.set_aspect('equal')
ax1.set_xlabel('$x$'); ax1.set_ylabel('$y$')

ax2.plot(y_u[:,0], y_u[:,1], 'steelblue', lw=1.5, label=f'Star 1 ($M={M1u}$)')
ax2.plot(y_u[:,2], y_u[:,3], 'tomato',    lw=1.5, label=f'Star 2 ($M={M2u}$)')
ax2.set_title('Unequal-mass binary'); ax2.legend(fontsize=9); ax2.set_aspect('equal')
ax2.set_xlabel('$x$'); ax2.set_ylabel('$y$')

ax3.semilogy(t_bin/T_bin, np.abs(E_bin/E_bin[0]-1)+1e-16, 'steelblue', lw=1.5, label='Equal mass')
ax3.semilogy(t_u/T_u,     np.abs(E_u/E_u[0]-1)+1e-16,     'tomato',    lw=1.5, label='Unequal mass')
ax3.set_xlabel('Orbits'); ax3.set_ylabel('|Energy error|')
ax3.set_title('Energy conservation'); ax3.legend()

plt.tight_layout()
plt.show()

### The Three-Body Problem

Adding a third body makes the system generally chaotic. The equations of motion for three bodies of masses $M_1, M_2, M_3$:

$$M_i\ddot{\mathbf{r}}_i = G M_i \sum_{j \neq i} \frac{M_j(\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}$$

State vector: 12 components ($3\times 2$ positions + $3\times 2$ velocities).

A beautiful exception: the **figure-eight orbit** (Chenciner & Montgomery, 2000) — three equal masses chasing each other on a figure-8.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: figure-eight orbit — three bodies tracing the figure-8 ────────
from scipy.integrate import solve_ivp

G_3b = 1.0

def three_body_rhs(masses):
    M1, M2, M3 = masses
    ms = [M1, M2, M3]
    def rhs(t, state):
        pos = [(state[2*i], state[2*i+1]) for i in range(3)]
        vel = [(state[6+2*i], state[6+2*i+1]) for i in range(3)]
        acc = [[0.0, 0.0] for _ in range(3)]
        for i in range(3):
            for j in range(3):
                if i != j:
                    dx = pos[j][0]-pos[i][0]; dy = pos[j][1]-pos[i][1]
                    r3 = (dx**2+dy**2)**1.5
                    acc[i][0] += G_3b*ms[j]*dx/r3
                    acc[i][1] += G_3b*ms[j]*dy/r3
        return np.array([vel[0][0], vel[0][1], vel[1][0], vel[1][1], vel[2][0], vel[2][1],
                         acc[0][0], acc[0][1], acc[1][0], acc[1][1], acc[2][0], acc[2][1]])
    return rhs

def three_body_energy(state, masses):
    M1,M2,M3 = masses
    pos = [state[:,2*i:2*i+2] for i in range(3)]
    vel = [state[:,6+2*i:6+2*i+2] for i in range(3)]
    ms = [M1,M2,M3]
    KE = sum(0.5*ms[i]*np.sum(vel[i]**2, axis=1) for i in range(3))
    r12 = np.sqrt(np.sum((pos[1]-pos[0])**2, axis=1))
    r13 = np.sqrt(np.sum((pos[2]-pos[0])**2, axis=1))
    r23 = np.sqrt(np.sum((pos[2]-pos[1])**2, axis=1))
    return KE - G_3b*(M1*M2/r12 + M1*M3/r13 + M2*M3/r23)

# Figure-eight orbit (Chenciner-Montgomery initial conditions)
masses_f8 = (1.0, 1.0, 1.0)
state0_f8 = np.array([
     0.97000436, -0.24308753,
    -0.97000436,  0.24308753,
     0.0,         0.0,
     0.93240737/2,  0.86473146/2,
     0.93240737/2,  0.86473146/2,
    -0.93240737,  -0.86473146,
])

T_f8 = 6.3259
rhs_f8 = three_body_rhs(masses_f8)
sol_f8 = solve_ivp(rhs_f8, (0, T_f8), state0_f8, method='DOP853', rtol=1e-11, atol=1e-13)
y_f8 = sol_f8.y.T; t_f8 = sol_f8.t
E_f8 = three_body_energy(y_f8, masses_f8)

h_anim_f8 = T_f8 / 200
t_anim_f8 = np.arange(0, T_f8 + h_anim_f8, h_anim_f8)
y_anim_f8 = np.zeros((len(t_anim_f8), 12))
y_anim_f8[0] = state0_f8.copy()
for i_step in range(1, len(t_anim_f8)):
    y_anim_f8[i_step] = rk4_step(rhs_f8, t_anim_f8[i_step-1], y_anim_f8[i_step-1], h_anim_f8)

with LivePlot(figsize=(8, 7)) as lp:
    lp.ax.set_xlim(-1.5, 1.5); lp.ax.set_ylim(-0.8, 0.8)
    lp.ax.set_xlabel('$x$'); lp.ax.set_ylabel('$y$')
    lp.ax.set_title('Figure-eight three-body orbit: RK4 steps')
    lp.ax.set_aspect('equal')
    
    f8_cols = ['steelblue', 'tomato', 'seagreen']
    trails = [lp.ax.plot([], [], color=c, lw=1.2, alpha=0.6)[0] for c in f8_cols]
    dots   = [lp.ax.plot([], [], 'o', color=c, ms=12, zorder=5, label=f'Body {i+1}')[0]
              for i, c in enumerate(f8_cols)]
    lp.ax.legend(fontsize=9)
    
    for i_step in range(1, len(t_anim_f8)):
        for k in range(3):
            trails[k].set_data(y_anim_f8[:i_step+1, 2*k], y_anim_f8[:i_step+1, 2*k+1])
            dots[k].set_data([y_anim_f8[i_step, 2*k]], [y_anim_f8[i_step, 2*k+1]])
        await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# Three-body problem
from scipy.integrate import solve_ivp

G_3b = 1.0

def three_body_rhs(masses):
    M1, M2, M3 = masses
    ms = [M1, M2, M3]
    def rhs(t, state):
        pos = [(state[2*i], state[2*i+1]) for i in range(3)]
        vel = [(state[6+2*i], state[6+2*i+1]) for i in range(3)]
        acc = [[0.0, 0.0] for _ in range(3)]
        for i in range(3):
            for j in range(3):
                if i != j:
                    dx = pos[j][0]-pos[i][0]; dy = pos[j][1]-pos[i][1]
                    r3 = (dx**2+dy**2)**1.5
                    acc[i][0] += G_3b*ms[j]*dx/r3
                    acc[i][1] += G_3b*ms[j]*dy/r3
        return np.array([vel[0][0], vel[0][1], vel[1][0], vel[1][1], vel[2][0], vel[2][1],
                         acc[0][0], acc[0][1], acc[1][0], acc[1][1], acc[2][0], acc[2][1]])
    return rhs

def three_body_energy(state, masses):
    M1,M2,M3 = masses
    pos = [state[:,2*i:2*i+2] for i in range(3)]
    vel = [state[:,6+2*i:6+2*i+2] for i in range(3)]
    ms = [M1,M2,M3]
    KE = sum(0.5*ms[i]*np.sum(vel[i]**2, axis=1) for i in range(3))
    r12 = np.sqrt(np.sum((pos[1]-pos[0])**2, axis=1))
    r13 = np.sqrt(np.sum((pos[2]-pos[0])**2, axis=1))
    r23 = np.sqrt(np.sum((pos[2]-pos[1])**2, axis=1))
    return KE - G_3b*(M1*M2/r12 + M1*M3/r13 + M2*M3/r23)

# Figure-eight orbit (Chenciner-Montgomery initial conditions)
masses_f8 = (1.0, 1.0, 1.0)
state0_f8 = np.array([
     0.97000436, -0.24308753,
    -0.97000436,  0.24308753,
     0.0,         0.0,
     0.93240737/2,  0.86473146/2,
     0.93240737/2,  0.86473146/2,
    -0.93240737,  -0.86473146,
])

T_f8 = 6.3259
rhs_f8 = three_body_rhs(masses_f8)
sol_f8 = solve_ivp(rhs_f8, (0, 2*T_f8), state0_f8, method='DOP853', rtol=1e-11, atol=1e-13)
y_f8 = sol_f8.y.T; t_f8 = sol_f8.t
E_f8 = three_body_energy(y_f8, masses_f8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cols_3b = ['steelblue', 'tomato', 'seagreen']
for i, col in enumerate(cols_3b):
    ax1.plot(y_f8[:, 2*i], y_f8[:, 2*i+1], color=col, lw=1.5, alpha=0.8, label=f'Body {i+1}')
    ax1.plot(y_f8[0, 2*i], y_f8[0, 2*i+1], 'o', color=col, ms=8, zorder=5)
ax1.set_xlabel('$x$'); ax1.set_ylabel('$y$')
ax1.set_title('Figure-eight three-body orbit')
ax1.legend(); ax1.set_aspect('equal')

ax2.semilogy(t_f8, np.abs(E_f8/E_f8[0]-1)+1e-16, 'steelblue', lw=1.5)
ax2.set_xlabel('Time'); ax2.set_ylabel('|Energy error|')
ax2.set_title('Figure-eight: energy conservation (DOP853)')
plt.tight_layout()
plt.show()

### N-Body Simulation with Leapfrog

The three-body equations extend to any number of bodies. For body $i$:
$$M_i\ddot{\mathbf{r}}_i = G M_i \sum_{j \neq i} \frac{M_j(\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3 + \epsilon^3}$$

The **softening parameter** $\epsilon$ prevents divergence when bodies pass close. We simulate a toy solar system: a central star plus four planets.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: solar toy system — integrate step by step as we plot ───────────

def nbody_accel(pos, masses, N_b, softening=1e-4):
    acc = np.zeros_like(pos)
    for i in range(N_b):
        for j in range(N_b):
            if i != j:
                dr = pos[j] - pos[i]
                r3 = (dr[0]**2 + dr[1]**2 + softening**2)**1.5
                acc[i] += G * masses[j] * dr / r3
    return acc

def nbody_energy_single(masses, pos, vel):
    """Energy for a single timestep. pos, vel: shape (N_bodies, 2)."""
    N = len(masses)
    KE = 0.5 * np.sum(masses * np.sum(vel**2, axis=-1))
    PE = 0.0
    for i in range(N):
        for j in range(i+1, N):
            r = np.linalg.norm(pos[j] - pos[i])
            PE -= G * masses[i] * masses[j] / r
    return KE + PE

# Setup
G = 1
M_star_s = 100.0
masses_ss = np.array([M_star_s, 1.0, 1.0, 1.0, 1.0])
radii_ss  = np.array([0.0, 1.0, 1.7, 2.6, 4.0])
v_circ_ss = np.sqrt(G * M_star_s / radii_ss[1:])
x0_ss = np.zeros((5, 2))
v0_ss = np.zeros((5, 2))

# calculate center-of-mass velocity to subtract off to prevent overall drift
net_v = np.sum(masses_ss[1:]*v_circ_ss)/np.sum(masses_ss)
v0_ss[0,1] = - net_v
for i in range(1, 5):
    x0_ss[i, 0] = radii_ss[i]
    v0_ss[i, 1] = v_circ_ss[i-1] - net_v

T_inner  = 2*np.pi*np.sqrt(radii_ss[1]**3 / (G * M_star_s))
T_sim_ss = 30 * T_inner
h_ss     = T_inner / 160
t_all    = np.arange(0, T_sim_ss + h_ss, h_ss)
N_steps  = len(t_all)
N_b      = len(masses_ss)

planet_cols_anim = ['goldenrod', 'steelblue', 'seagreen', 'tomato', 'darkorange']
planet_sizes     = [20, 10, 10, 10, 10]
labels_anim      = ['Star', 'Planet 1', 'Planet 2', 'Planet 3', 'Planet 4']

# Render every render_every steps so animation isn't too slow
render_every = max(1, N_steps // 300)

with LivePlot(figsize=(7, 7)) as lp:
    r_max = radii_ss[-1] * 1.3
    lp.ax.set_xlim(-r_max, r_max); lp.ax.set_ylim(-r_max, r_max)
    lp.ax.set_xlabel('$x$'); lp.ax.set_ylabel('$y$')
    lp.ax.set_title('N-body solar system (Leapfrog) — integrating live')
    lp.ax.set_aspect('equal')

    trails_ss = [lp.ax.plot([], [], color=planet_cols_anim[i], lw=0.8, alpha=0.5,
                            label=labels_anim[i])[0] for i in range(N_b)]
    dots_ss   = [lp.ax.plot([], [], 'o', color=planet_cols_anim[i],
                            ms=planet_sizes[i], zorder=5)[0] for i in range(N_b)]
    time_text = lp.ax.text(0.02, 0.97, '', transform=lp.ax.transAxes, va='top', fontsize=10)
    lp.ax.legend(fontsize=8, loc='upper right')

    # Integration state
    x_cur = x0_ss.copy()
    v_cur = v0_ss.copy()
    a_cur = nbody_accel(x_cur, masses_ss, N_b)
    E0    = nbody_energy_single(masses_ss, x_cur, v_cur)

    # Rolling trail buffer: keep last 60 positions per body
    TRAIL = 60
    trail_buf = np.full((TRAIL, N_b, 2), np.nan)
    buf_ptr   = 0

    for step in range(N_steps):
        # Leapfrog kick-drift-kick
        v_half = v_cur + 0.5*h_ss * a_cur
        x_cur  = x_cur + h_ss * v_half
        a_cur  = nbody_accel(x_cur, masses_ss, N_b)
        v_cur  = v_half + 0.5*h_ss * a_cur

        # Store in rolling buffer
        trail_buf[buf_ptr % TRAIL] = x_cur
        buf_ptr += 1

        if step % render_every == 0:
            # Build ordered trail (oldest first)
            if buf_ptr < TRAIL:
                ordered = trail_buf[:buf_ptr]
            else:
                idx = np.arange(buf_ptr, buf_ptr + TRAIL) % TRAIL
                ordered = trail_buf[idx]

            for i in range(N_b):
                trails_ss[i].set_data(ordered[:, i, 0], ordered[:, i, 1])
                dots_ss[i].set_data([x_cur[i, 0]], [x_cur[i, 1]])

            E_cur = nbody_energy_single(masses_ss, x_cur, v_cur)
            t_cur = t_all[step]
            time_text.set_text(
                f't = {t_cur/T_inner:.1f} orbits  |  ΔE/E₀ = {abs(E_cur/E0 - 1):.1e}')
            await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# N-body leapfrog simulation

def nbody_leapfrog(masses, t_span, x0, v0, h, softening=1e-4):
    t0, tf = t_span
    t = np.arange(t0, tf+h, h)
    N_steps = len(t); N_b = len(masses)
    x = np.zeros((N_steps, N_b, 2))
    v = np.zeros((N_steps, N_b, 2))
    x[0] = x0.copy(); v[0] = v0.copy()

    def accel(pos):
        acc = np.zeros_like(pos)
        for i in range(N_b):
            for j in range(N_b):
                if i != j:
                    dr = pos[j] - pos[i]
                    r3 = (dr[0]**2 + dr[1]**2 + softening**2)**1.5
                    acc[i] += G * masses[j] * dr / r3
        return acc

    a_curr = accel(x[0])
    for step in range(N_steps - 1):
        v_half    = v[step] + 0.5*h*a_curr
        x[step+1] = x[step] + h*v_half
        a_curr    = accel(x[step+1])
        v[step+1] = v_half + 0.5*h*a_curr
    return t, x, v

def nbody_energy(masses, positions, velocities):
    # positions, velocities: shape (N_steps, N_bodies, 2)
    N = len(masses)
    # sum over xy (axis=-1) -> (N_steps, N_bodies); weight by mass -> sum over bodies (axis=1)
    KE = 0.5*np.sum(masses[None,:]*np.sum(velocities**2, axis=-1), axis=1)
    PE = np.zeros(len(positions))
    for i in range(N):
        for j in range(i+1, N):
            dr = positions[:,j] - positions[:,i]
            r  = np.sqrt(np.sum(dr**2, axis=-1))
            PE -= G*masses[i]*masses[j]/r
    return KE + PE

# Solar system toy model
G = 1
M_star_s = 100.0
masses_ss = np.array([M_star_s, 1.0, 1.0, 1.0, 1.0])
radii_ss  = np.array([0.0, 1.0, 1.7, 2.6, 4.0])
v_circ_ss = np.sqrt(G*M_star_s/radii_ss[1:])
x0_ss = np.zeros((5, 2))
v0_ss = np.zeros((5, 2))
# calculate center-of-mass velocity to subtract off to prevent overall drift
net_v = np.sum(masses_ss[1:]*v_circ_ss)/np.sum(masses_ss)
v0_ss[0,1] = - net_v
for i in range(1, 5):
    x0_ss[i, 0] = radii_ss[i]
    v0_ss[i, 1] = v_circ_ss[i-1] - net_v

T_inner = 2*np.pi*np.sqrt(radii_ss[1]**3/(G*M_star_s))
T_sim_ss = 30*T_inner
h_ss = T_inner/160

print("Running N-body simulation...")
t_ss, x_ss, v_ss = nbody_leapfrog(masses_ss, (0, T_sim_ss), x0_ss, v0_ss, h_ss)
E_ss = nbody_energy(masses_ss, x_ss, v_ss)
print(f"Done. {len(t_ss)} steps, max energy error: {np.max(np.abs(E_ss/E_ss[0]-1)):.2e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

planet_cols = ['goldenrod', 'steelblue', 'seagreen', 'tomato', 'darkorange']
labels_ss   = ['Star', 'Planet 1', 'Planet 2', 'Planet 3', 'Planet 4']
for i in range(5):
    ax1.plot(x_ss[:,i,0], x_ss[:,i,1], color=planet_cols[i], lw=0.6+(i==0)*0.6,
             alpha=0.8, label=labels_ss[i])
ax1.set_xlabel('$x$'); ax1.set_ylabel('$y$')
ax1.set_title('5-body solar system toy (Leapfrog)')
ax1.legend(fontsize=8); ax1.set_aspect('equal')

ax2.semilogy(t_ss/T_inner, np.abs(E_ss/E_ss[0]-1)+1e-16, 'steelblue', lw=1.0)
ax2.set_xlabel('Inner orbital periods'); ax2.set_ylabel('|Energy error|')
ax2.set_title('N-body energy conservation')
plt.tight_layout()
plt.show()

### N-Body Solar System: RK4 vs Leapfrog

RK4 is a higher-order method ($p=4$) than leapfrog ($p=2$), but it is **not symplectic**.
For long-time Hamiltonian systems this matters: RK4 allows a slow secular energy drift,
while leapfrog's energy error stays bounded. The comparison below uses identical initial
conditions and the same step size $h = T_{\rm inner}/80$ so any difference is due to the
integrator alone, not the resolution.

In [ ]:
plt.close('all')
%matplotlib widget

# ── ANIMATION: solar toy system with RK4 — integrate step by step as we plot ─

def nbody_rhs(state, masses, N_b, softening=1e-4):
    """RHS for N-body system packed as state = [x0,y0, x1,y1, ..., vx0,vy0, ...]
    Returns d/dt(state) with same layout."""
    pos = state[:N_b*2].reshape(N_b, 2)
    vel = state[N_b*2:].reshape(N_b, 2)
    acc = np.zeros((N_b, 2))
    for i in range(N_b):
        for j in range(N_b):
            if i != j:
                dr = pos[j] - pos[i]
                r3 = (dr[0]**2 + dr[1]**2 + softening**2)**1.5
                acc[i] += G * masses[j] * dr / r3
    return np.concatenate([vel.ravel(), acc.ravel()])

# Same setup as the leapfrog cells
N_b_rk = len(masses_ss)
render_every_rk = max(1, N_steps // 300)

with LivePlot(figsize=(7, 7)) as lp:
    r_max = radii_ss[-1] * 1.3
    lp.ax.set_xlim(-r_max, r_max); lp.ax.set_ylim(-r_max, r_max)
    lp.ax.set_xlabel('$x$'); lp.ax.set_ylabel('$y$')
    lp.ax.set_title('N-body solar system (RK4) — integrating live')
    lp.ax.set_aspect('equal')

    planet_cols_rk = ['goldenrod', 'steelblue', 'seagreen', 'tomato', 'darkorange']
    planet_sizes   = [20, 10, 10, 10, 10]
    labels_rk      = ['Star', 'Planet 1', 'Planet 2', 'Planet 3', 'Planet 4']

    trails_rk = [lp.ax.plot([], [], color=planet_cols_rk[i], lw=0.8, alpha=0.5,
                            label=labels_rk[i])[0] for i in range(N_b_rk)]
    dots_rk   = [lp.ax.plot([], [], 'o', color=planet_cols_rk[i],
                            ms=planet_sizes[i], zorder=5)[0] for i in range(N_b_rk)]
    time_text_rk = lp.ax.text(0.02, 0.97, '', transform=lp.ax.transAxes,
                              va='top', fontsize=10)
    lp.ax.legend(fontsize=8, loc='upper right')

    # Pack initial state as flat vector for RK4
    state_rk = np.concatenate([x0_ss.ravel(), v0_ss.ravel()])
    E0_rk = nbody_energy_single(masses_ss, x0_ss, v0_ss)

    TRAIL = 60
    trail_buf_rk = np.full((TRAIL, N_b_rk, 2), np.nan)
    buf_ptr_rk   = 0

    for step in range(N_steps):
        # RK4 step
        k1 = nbody_rhs(state_rk,              masses_ss, N_b_rk)
        k2 = nbody_rhs(state_rk + h_ss/2*k1,  masses_ss, N_b_rk)
        k3 = nbody_rhs(state_rk + h_ss/2*k2,  masses_ss, N_b_rk)
        k4 = nbody_rhs(state_rk + h_ss*k3,    masses_ss, N_b_rk)
        state_rk = state_rk + (h_ss/6)*(k1 + 2*k2 + 2*k3 + k4)

        pos_rk = state_rk[:N_b_rk*2].reshape(N_b_rk, 2)
        vel_rk = state_rk[N_b_rk*2:].reshape(N_b_rk, 2)

        trail_buf_rk[buf_ptr_rk % TRAIL] = pos_rk
        buf_ptr_rk += 1

        if step % render_every_rk == 0:
            if buf_ptr_rk < TRAIL:
                ordered_rk = trail_buf_rk[:buf_ptr_rk]
            else:
                idx = np.arange(buf_ptr_rk, buf_ptr_rk + TRAIL) % TRAIL
                ordered_rk = trail_buf_rk[idx]

            for i in range(N_b_rk):
                trails_rk[i].set_data(ordered_rk[:, i, 0], ordered_rk[:, i, 1])
                dots_rk[i].set_data([pos_rk[i, 0]], [pos_rk[i, 1]])

            E_cur_rk = nbody_energy_single(masses_ss, pos_rk, vel_rk)
            time_text_rk.set_text(
                f't = {t_all[step]/T_inner:.1f} orbits  |  ΔE/E₀ = {abs(E_cur_rk/E0_rk - 1):.1e}')
            await lp.update()


In [ ]:
plt.close('all')
%matplotlib inline

# N-body RK4 simulation — same parameters as leapfrog for direct comparison

def nbody_rk4_integrate(masses, t_span, x0, v0, h, softening=1e-4):
    """Full RK4 integrator for N-body system. Returns t, x, v arrays."""
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    N_steps_r = len(t); N_b_r = len(masses)
    x = np.zeros((N_steps_r, N_b_r, 2))
    v = np.zeros((N_steps_r, N_b_r, 2))
    x[0] = x0.copy(); v[0] = v0.copy()

    def rhs(pos_flat, vel_flat):
        pos = pos_flat.reshape(N_b_r, 2)
        acc = np.zeros((N_b_r, 2))
        for i in range(N_b_r):
            for j in range(N_b_r):
                if i != j:
                    dr = pos[j] - pos[i]
                    r3 = (dr[0]**2 + dr[1]**2 + softening**2)**1.5
                    acc[i] += G * masses[j] * dr / r3
        return vel_flat.copy(), acc.ravel()

    for s in range(N_steps_r - 1):
        xf = x[s].ravel(); vf = v[s].ravel()
        dx1, dv1 = rhs(xf,              vf             )
        dx2, dv2 = rhs(xf + h/2*dx1,   vf + h/2*dv1  )
        dx3, dv3 = rhs(xf + h/2*dx2,   vf + h/2*dv2  )
        dx4, dv4 = rhs(xf + h*dx3,     vf + h*dv3    )
        x[s+1] = (xf + (h/6)*(dx1 + 2*dx2 + 2*dx3 + dx4)).reshape(N_b_r, 2)
        v[s+1] = (vf + (h/6)*(dv1 + 2*dv2 + 2*dv3 + dv4)).reshape(N_b_r, 2)
    return t, x, v

print('Running N-body RK4 simulation...')
t_rk, x_rk, v_rk = nbody_rk4_integrate(masses_ss, (0, T_sim_ss), x0_ss, v0_ss, h_ss)
E_rk = nbody_energy(masses_ss, x_rk, v_rk)
print(f'Done. {len(t_rk)} steps, max energy error: {np.max(np.abs(E_rk/E_rk[0]-1)):.2e}')

print('Running N-body Leapfrog simulation...')
t_lf2, x_lf2, v_lf2 = nbody_leapfrog(masses_ss, (0, T_sim_ss), x0_ss, v0_ss, h_ss)
E_lf2 = nbody_energy(masses_ss, x_lf2, v_lf2)
print(f'Done. {len(t_lf2)} steps, max energy error: {np.max(np.abs(E_lf2/E_lf2[0]-1)):.2e}')

planet_cols = ['goldenrod', 'steelblue', 'seagreen', 'tomato', 'darkorange']
labels_ss   = ['Star', 'Planet 1', 'Planet 2', 'Planet 3', 'Planet 4']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Orbits: Leapfrog
for i in range(5):
    axes[0][0].plot(x_lf2[:,i,0], x_lf2[:,i,1], color=planet_cols[i],
                    lw=0.6+(i==0)*0.6, alpha=0.8, label=labels_ss[i])
axes[0][0].set_xlabel('$x$'); axes[0][0].set_ylabel('$y$')
axes[0][0].set_title('Leapfrog — orbital trajectories')
axes[0][0].legend(fontsize=8); axes[0][0].set_aspect('equal')

# Orbits: RK4
for i in range(5):
    axes[0][1].plot(x_rk[:,i,0], x_rk[:,i,1], color=planet_cols[i],
                    lw=0.6+(i==0)*0.6, alpha=0.8, label=labels_ss[i])
axes[0][1].set_xlabel('$x$'); axes[0][1].set_ylabel('$y$')
axes[0][1].set_title('RK4 — orbital trajectories')
axes[0][1].legend(fontsize=8); axes[0][1].set_aspect('equal')

# Energy: both on same axes
axes[1][0].semilogy(t_lf2/T_inner, np.abs(E_lf2/E_lf2[0]-1)+1e-16,
                    'steelblue', lw=1.0, label='Leapfrog')
axes[1][0].semilogy(t_rk/T_inner,  np.abs(E_rk/E_rk[0]-1)+1e-16,
                    'tomato',    lw=1.0, label='RK4')
axes[1][0].set_xlabel('Inner orbital periods'); axes[1][0].set_ylabel('|Energy error|')
axes[1][0].set_title('Energy conservation: Leapfrog vs RK4 (same $h$)')
axes[1][0].legend()

# Outermost planet orbit close-up: leapfrog vs RK4 overlay
axes[1][1].plot(x_lf2[:,4,0], x_lf2[:,4,1], 'steelblue', lw=1.0, alpha=0.8,
                label='Leapfrog')
axes[1][1].plot(x_rk[:,4,0],  x_rk[:,4,1],  'tomato',    lw=1.0, alpha=0.8,
                label='RK4')
axes[1][1].set_xlabel('$x$'); axes[1][1].set_ylabel('$y$')
axes[1][1].set_title('Outermost planet orbit: Leapfrog vs RK4')
axes[1][1].legend(); axes[1][1].set_aspect('equal')

plt.suptitle(f'N-body solar toy: Leapfrog vs RK4  ($h = T_{{\\rm inner}}/80$, '
             f'{len(t_rk)} steps, {T_sim_ss/T_inner:.0f} inner orbits)',
             fontsize=12)
plt.tight_layout()
plt.show()


---
## Summary: Choosing an Integrator

| Method | Order | f-evals/step | Energy conserv. | Best for |
|---|---|---|---|---|
| Euler | 1 | 1 | Drifts (grows) | Learning/demos only |
| Heun / RK2 | 2 | 2 | Drifts slowly | Quick estimates |
| Classical RK4 | 4 | 4 | Very good | General purpose |
| Leapfrog/Verlet | 2 | 1 | Bounded (symplectic) | Long Hamiltonian sims |
| `solve_ivp` RK45 | 4–5 | 6 | Very good | Smooth non-stiff ODEs |
| `solve_ivp` DOP853 | 8 | 13 | Excellent | High-accuracy |
| `solve_ivp` Radau | variable | varies | Good | **Stiff** systems |

**Rules of thumb:**
1. For one-off physics calculations, start with RK4.
2. For long-time Hamiltonian systems (orbits, MD), use Leapfrog.
3. For production code, use `scipy.integrate.solve_ivp`.
4. **Always monitor a conservation law** as a sanity check.
5. **Halve the step size and rerun** to verify your results have converged.